# FreshScan on Google Colab

Runs the full FreshScan replay experiment on KITTI raw. The code from the repository is written into the runtime by the cells below, so nothing has to be uploaded.

**Usage**
1. Choose a GPU runtime (*Runtime > Change runtime type > T4 GPU*), then *Runtime > Run all*.
2. Results go to `MyDrive/FreshScan/`. If Colab disconnects, run all again; finished drives are skipped.

**Data.** KITTI raw requires accepting its license at <https://www.cvlibs.net/datasets/kitti/raw_data.php>. Drives are downloaded one at a time, only the LiDAR and OXTS files are unpacked, and the scans are deleted after each drive, so the Colab disk does not fill up. If the download fails, put the *synced+rectified data* zips in `MyDrive/FreshScan/kitti_zips/` and the notebook will use them.

**GPU.** ICP takes most of the time. On a GPU runtime (T4) it runs on the GPU through Open3D; cell 5 first checks that GPU and CPU give the same transform. Results are identical on CPU, only slower. Use one runtime type for the whole run so the timing numbers come from one machine (recorded in `run_info/`).

**Checkpoint.** There is no trained model. The calibration on normal driving (scan period, 99th percentiles of the checks, thresholds) is saved in `checkpoint/`. With the per-frame files in `features/`, cell 10 re-evaluates other thresholds in seconds.

In [ ]:
#@title 1. Settings  { display-mode: "form" }
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "FreshScan"  #@param {type:"string"}
#@markdown Delete each drive's LiDAR scans after it is processed (saves disk space):
DELETE_SCANS_AFTER_EACH_DRIVE = True  #@param {type:"boolean"}
#@markdown ICP device: `auto` = GPU if available, else CPU (same result, different speed)
ICP_BACKEND = "auto"  #@param ["auto", "cuda", "open3d", "numpy"]

In [ ]:
#@title 2. Folders, Google Drive and a helper to run scripts
import os, sys, glob, shutil, subprocess, time

ROOT  = "/content/freshscan"
CODE  = f"{ROOT}/code"
KITTI = "/content/kitti_raw"          # raw scans stay on the Colab disk (too large for Drive)
ZIPS  = "/content/zips"
IN_COLAB = "google.colab" in sys.modules

if SAVE_TO_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
else:
    OUT = f"/content/{DRIVE_FOLDER}"   # results are lost when the Colab runtime is reset

RESULTS = f"{OUT}/results"
for d in [CODE, KITTI, ZIPS, RESULTS]:
    os.makedirs(d, exist_ok=True)
for sub in ["freshscan", "scripts", "configs"]:
    os.makedirs(f"{CODE}/{sub}", exist_ok=True)

def sh(cmd, cwd=CODE):
    """Run a shell command and show its output live (plain subprocess output is hidden in Colab)."""
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {p.returncode}): {cmd}")

print("Results:", RESULTS)

In [ ]:
#@title 3. Install Open3D (fast scan matching; the code falls back to NumPy without it)
try:
    import open3d  # noqa: F401
except ImportError:
    sh(f"{sys.executable} -m pip install -q open3d", cwd="/content")
try:
    import open3d
    print("ICP backend: Open3D", open3d.__version__)
except Exception as e:
    print("Open3D is not available, using the slower NumPy ICP:", e)
import numpy, scipy, pandas, sklearn, matplotlib, yaml
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__)
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 and gpu.stdout.strip() else "none (CPU runtime)")
try:
    print("Open3D can use the GPU:", open3d.core.cuda.is_available())
except Exception:
    print("Open3D can use the GPU: False")

## 4. FreshScan code
The next cells write the repository code into `/content/freshscan/code`. They do not need to be changed.

In [ ]:
%%writefile {CODE}/freshscan/__init__.py
"""FreshScan: detecting replayed LiDAR frames with ego-motion consistency.

Modules
-------
kitti_loader : read KITTI raw synced drives (Velodyne frames, OXTS motion, LiDAR timestamps)
replay       : replay injection (sigma(t) = t - R inside random windows) with frame labels
registration : C1 preparation (crop + voxel) and C2 LiDAR ego-motion by point-to-point ICP
features     : C3 consistency checks (motion residuals and seam features)
detector     : C5 decision (benign-normalized max score, m-of-n alarm, trust) and B1 timestamp check
"""

__version__ = "1.0.0"

In [ ]:
%%writefile {CODE}/freshscan/kitti_loader.py
"""Reader for KITTI raw *synced* drives.

Expected layout (as unpacked from the KITTI raw data zips):

    <root>/<date>/<date>_drive_<id>_sync/
        velodyne_points/data/0000000000.bin   float32 x, y, z, reflectance
        velodyne_points/timestamps.txt        one timestamp per frame (optional)
        oxts/data/0000000000.txt              30 values per line

OXTS fields used (0-based index in each line, see the KITTI raw devkit readme):
    8  vf : forward velocity, parallel to the earth surface [m/s]
    22 wu : angular rate around the upward axis [rad/s]
"""

from __future__ import annotations

import glob
import os
from dataclasses import dataclass
from datetime import datetime

import numpy as np

OXTS_VF = 8
OXTS_WU = 22


@dataclass
class Drive:
    name: str
    velo_files: list
    speed: np.ndarray        # reference forward speed per frame [m/s]
    yaw_rate: np.ndarray     # reference yaw rate per frame [rad/s]
    stamps: np.ndarray       # LiDAR header time per frame [s], relative to frame 0

    def __len__(self) -> int:
        return len(self.velo_files)

    def frame(self, i: int) -> np.ndarray:
        """Frame i as an (N, 3) array of x, y, z points (Velodyne axes: x forward, y left, z up)."""
        pts = np.fromfile(self.velo_files[i], dtype=np.float32).reshape(-1, 4)
        return pts[:, :3].astype(np.float64)


def _read_stamps(path: str, n: int, period: float) -> np.ndarray:
    if not os.path.exists(path):
        return np.arange(n) * period
    out = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            # format: 2011-09-26 13:02:25.964389445  (nanoseconds are cut to microseconds)
            day, clock = line.split(" ")
            hms, frac = clock.split(".")
            t = datetime.strptime(f"{day} {hms}", "%Y-%m-%d %H:%M:%S").timestamp() + float("0." + frac)
            out.append(t)
    out = np.asarray(out[:n])
    return out - out[0]


def load_drive(root: str, name: str, period: float = 0.1) -> Drive:
    """`name` is e.g. '2011_09_30_drive_0018'."""
    date = name[:10]
    base = os.path.join(root, date, f"{name}_sync")
    velo = sorted(glob.glob(os.path.join(base, "velodyne_points", "data", "*.bin")))
    oxts = sorted(glob.glob(os.path.join(base, "oxts", "data", "*.txt")))
    if not velo or len(velo) != len(oxts):
        raise FileNotFoundError(f"No matching Velodyne/OXTS files under {base} "
                                f"({len(velo)} velodyne, {len(oxts)} oxts)")
    vals = np.array([np.loadtxt(f) for f in oxts])
    stamps = _read_stamps(os.path.join(base, "velodyne_points", "timestamps.txt"), len(velo), period)
    return Drive(name=name, velo_files=velo, speed=vals[:, OXTS_VF], yaw_rate=vals[:, OXTS_WU],
                 stamps=stamps)

In [ ]:
%%writefile {CODE}/freshscan/replay.py
"""Replay injection.

Inside a window [t_s, t_e], slot t delivers frame L_{t-R}. The header timestamp and the
IMU/GPS reference stay those of slot t. Outside windows, sigma(t) = t.

Labels
------
label = 1 for every slot inside a window and for the first fresh slot after it
(the exit seam), because that slot's registration still spans the last replayed frame.
seam  = 1 for the entry slot t_s and the exit slot t_e + 1.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np


@dataclass
class ReplayConfig:
    replay_s: float = 10.0     # replay age R [s]
    win_min_s: float = 2.0     # shortest window [s]
    win_max_s: float = 10.0    # longest window [s]
    gap_s: float = 10.0        # benign gap between windows [s]
    period_s: float = 0.1      # LiDAR period Delta [s]
    seed: int = 0


def windows(n: int, cfg: ReplayConfig) -> list:
    """Random, non-overlapping windows (start, end) inclusive, with start >= R."""
    rng = np.random.default_rng(cfg.seed)
    R = int(round(cfg.replay_s / cfg.period_s))
    wmin = int(round(cfg.win_min_s / cfg.period_s))
    wmax = int(round(cfg.win_max_s / cfg.period_s))
    gap = int(round(cfg.gap_s / cfg.period_s))
    out, t = [], max(R, gap)
    while True:
        start = t + int(rng.integers(0, gap + 1))
        length = int(rng.integers(wmin, wmax + 1))
        end = start + length - 1
        if end + 2 >= n:          # keep room for the exit seam
            break
        out.append((start, end))
        t = end + 1 + gap
    return out


def inject(n: int, cfg: ReplayConfig):
    """Return sigma, label, seam, R (in frames), and the list of windows."""
    R = int(round(cfg.replay_s / cfg.period_s))
    sigma = np.arange(n)
    label = np.zeros(n, dtype=int)
    seam = np.zeros(n, dtype=int)
    wins = windows(n, cfg)
    for s, e in wins:
        t = np.arange(s, e + 1)
        sigma[t] = t - R
        label[s:e + 2] = 1          # window + exit seam
        seam[s] = 1
        seam[e + 1] = 1
    return sigma, label, seam, R, wins

In [ ]:
%%writefile {CODE}/freshscan/registration.py
"""LiDAR ego-motion by point-to-point ICP (component C2).

The transform maps the *current* frame P_t onto the *previous* frame P_{t-1}.
Because static points move opposite to the vehicle, this transform equals the
vehicle motion between the two frames, expressed in the frame of P_{t-1}
(KITTI Velodyne axes: x forward, y left, z up).

Backends (same algorithm and result, chosen with select_backend()):
    cuda    Open3D tensor ICP on the GPU (Open3D wheels on PyPI are built with CUDA)
    open3d  Open3D ICP on the CPU
    numpy   small NumPy/SciPy ICP, runs anywhere (slowest)
"auto" picks cuda if a GPU is visible to Open3D, then open3d, then numpy.

Principle P2: the initial guess comes only from the previous LiDAR estimate,
never from the IMU/GPS reference.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

try:  # optional fast backend
    import open3d as o3d  # type: ignore
    HAVE_O3D = True
except Exception:  # pragma: no cover - depends on the environment
    HAVE_O3D = False

from scipy.spatial import cKDTree


@dataclass
class IcpResult:
    T: np.ndarray        # 4x4 vehicle motion (P_t -> P_{t-1})
    fitness: float       # share of points in P_t with a match closer than d_max
    rmse: float          # inlier RMSE [m]

    @property
    def forward(self) -> float:
        return float(self.T[0, 3])

    @property
    def yaw(self) -> float:
        return float(np.arctan2(self.T[1, 0], self.T[0, 0]))


def prepare(points: np.ndarray, r_min: float = 3.0, r_max: float = 60.0,
            voxel: float = 0.5) -> np.ndarray:
    """Crop to [r_min, r_max] (horizontal range) and voxel down-sample."""
    r = np.linalg.norm(points[:, :2], axis=1)
    pts = points[(r >= r_min) & (r <= r_max)]
    if voxel <= 0 or len(pts) == 0:
        return pts
    # One integer key per voxel; np.unique on a 1-D key is several times faster than on rows.
    k = np.floor(pts / voxel).astype(np.int64)
    k -= k.min(axis=0)
    dims = k.max(axis=0) + 1
    key = (k[:, 0] * dims[1] + k[:, 1]) * dims[2] + k[:, 2]
    _, idx = np.unique(key, return_index=True)
    return pts[np.sort(idx)]


def _best_fit(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Least-squares rigid transform mapping a onto b (Kabsch)."""
    ca, cb = a.mean(0), b.mean(0)
    H = (a - ca).T @ (b - cb)
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        R = Vt.T @ U.T
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = cb - R @ ca
    return T


def _icp_numpy(src: np.ndarray, dst: np.ndarray, init: np.ndarray,
               d_max: float, iters: int, tol: float) -> IcpResult:
    tree = cKDTree(dst)
    T = init.copy()
    prev_err = np.inf
    for _ in range(iters):
        moved = src @ T[:3, :3].T + T[:3, 3]
        dist, j = tree.query(moved, distance_upper_bound=d_max)
        ok = np.isfinite(dist)
        if ok.sum() < 10:
            break
        T = _best_fit(src[ok], dst[j[ok]])
        err = float(np.sqrt(np.mean(dist[ok] ** 2)))
        if abs(prev_err - err) < tol:
            break
        prev_err = err
    moved = src @ T[:3, :3].T + T[:3, 3]
    dist, _ = tree.query(moved, distance_upper_bound=d_max)
    ok = np.isfinite(dist)
    fitness = float(ok.mean()) if len(ok) else 0.0
    rmse = float(np.sqrt(np.mean(dist[ok] ** 2))) if ok.any() else float("inf")
    return IcpResult(T=T, fitness=fitness, rmse=rmse)


BACKEND = "open3d" if HAVE_O3D else "numpy"
_CUDA = None


def cuda_available() -> bool:
    try:
        return bool(HAVE_O3D and o3d.core.cuda.is_available())
    except Exception:  # pragma: no cover
        return False


def select_backend(pref: str = "auto") -> str:
    """Choose the ICP backend ('auto', 'cuda', 'open3d' or 'numpy') and return the one in use."""
    global BACKEND, _CUDA
    pref = (pref or "auto").lower()
    if pref in ("auto", "cuda") and cuda_available():
        BACKEND, _CUDA = "cuda", o3d.core.Device("CUDA:0")
    elif pref == "numpy" or not HAVE_O3D:
        BACKEND = "numpy"
    else:
        if pref == "cuda":
            print("[FreshScan] CUDA requested but not available to Open3D; using the CPU.")
        BACKEND = "open3d"
    return BACKEND


def _icp_cuda(cur, prev, init, d_max, iters) -> IcpResult:
    reg_t = o3d.t.pipelines.registration
    s = o3d.t.geometry.PointCloud(o3d.core.Tensor(np.ascontiguousarray(cur, dtype=np.float32), device=_CUDA))
    d = o3d.t.geometry.PointCloud(o3d.core.Tensor(np.ascontiguousarray(prev, dtype=np.float32), device=_CUDA))
    res = reg_t.icp(s, d, d_max, o3d.core.Tensor(np.asarray(init, dtype=np.float64)),
                    reg_t.TransformationEstimationPointToPoint(),
                    reg_t.ICPConvergenceCriteria(max_iteration=iters))
    return IcpResult(T=res.transformation.cpu().numpy().astype(np.float64),
                     fitness=float(res.fitness), rmse=float(res.inlier_rmse))


def icp(cur: np.ndarray, prev: np.ndarray, init: np.ndarray | None = None,
        d_max: float = 1.0, iters: int = 50, tol: float = 1e-5) -> IcpResult:
    """Register the current frame `cur` onto the previous frame `prev`."""
    init = np.eye(4) if init is None else init
    if BACKEND == "cuda":
        return _icp_cuda(cur, prev, init, d_max, iters)
    if BACKEND == "open3d":
        s = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(cur))
        d = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(prev))
        reg = o3d.pipelines.registration.registration_icp(
            s, d, d_max, init,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=iters),
        )
        return IcpResult(T=np.asarray(reg.transformation), fitness=float(reg.fitness),
                         rmse=float(reg.inlier_rmse))
    return _icp_numpy(cur, prev, init, d_max, iters, tol)

In [ ]:
%%writefile {CODE}/freshscan/features.py
"""C3 consistency checks.

Feature vector z_t = (r_v, r_w, 1 - f, e):
    r_v  relative speed residual  |v_hat - v_bar| / (|v_bar| + v0)
    r_w  yaw-rate residual        |w_hat - w_bar|
    1-f  ICP misfit (seam)        1 - fitness
    e    ICP inlier RMSE (seam)

motion_features() is the version used for evaluation: LiDAR rates use the sensor's real
scan period (a constant measured on benign calibration drives, never per-frame timestamps),
both motion sources are averaged over the last k slots (causal), and the motion checks
abstain when the reference speed is below v_min (principle P3: too little motion to judge).
"""

from __future__ import annotations

import numpy as np

FEATURES = ("r_v", "r_w", "one_minus_fit", "rmse")
SEAM_FEATURES = ("one_minus_fit", "rmse")          # baseline B2
RESIDUAL_FEATURES = ("r_v", "r_w")                 # baseline B3


def residuals(v_hat, w_hat, v_bar, w_bar, v0: float = 1.0):
    v_hat, w_hat, v_bar, w_bar = map(np.asarray, (v_hat, w_hat, v_bar, w_bar))
    r_v = np.abs(v_hat - v_bar) / (np.abs(v_bar) + v0)
    r_w = np.abs(w_hat - w_bar)
    return r_v, r_w


def feature_table(v_hat, w_hat, v_bar, w_bar, fit, rmse, v0: float = 1.0) -> dict:
    r_v, r_w = residuals(v_hat, w_hat, v_bar, w_bar, v0)
    return {"r_v": r_v, "r_w": r_w,
            "one_minus_fit": 1.0 - np.asarray(fit, dtype=float),
            "rmse": np.asarray(rmse, dtype=float)}


def causal_mean(x, k: int) -> np.ndarray:
    """Mean of the last k values (fewer at the start of a stream)."""
    x = np.asarray(x, dtype=float)
    if k <= 1:
        return x
    c = np.cumsum(np.r_[0.0, x])
    idx = np.arange(1, len(x) + 1)
    lo = np.maximum(idx - k, 0)
    return (c[idx] - c[lo]) / (idx - lo)


def motion_features(v_hat, w_hat, v_bar, w_bar, period_ratio: float = 1.0, k: int = 1,
                    v0: float = 1.0, v_min: float = 0.0, w0: float | None = None):
    """Speed and yaw-rate residuals with period correction, k-slot averaging and low-speed abstain.

    period_ratio = nominal period used by step 1 / real scan period.
    w0: if given, the yaw residual is relative, |w_hat - w_bar| / (|w_bar| + w0), like the speed residual.
    Returns r_v, r_w and a boolean array `checked` (False where the check abstains).
    """
    vh = causal_mean(np.asarray(v_hat, dtype=float) * period_ratio, k)
    wh = causal_mean(np.asarray(w_hat, dtype=float) * period_ratio, k)
    vb, wb = causal_mean(v_bar, k), causal_mean(w_bar, k)
    r_v = np.abs(vh - vb) / (np.abs(vb) + v0)
    r_w = np.abs(wh - wb)
    if w0 is not None:
        r_w = r_w / (np.abs(wb) + w0)
    checked = np.abs(vb) >= v_min
    r_v = np.where(checked, r_v, 0.0)
    r_w = np.where(checked, r_w, 0.0)
    return r_v, r_w, checked

In [ ]:
%%writefile {CODE}/freshscan/detector.py
"""C5 decision and the B1 timestamp baseline.

All statistics are fit on benign calibration drives only.
"""

from __future__ import annotations

import numpy as np


def _clean(Z: np.ndarray) -> np.ndarray:
    return np.nan_to_num(np.asarray(Z, dtype=float), nan=0.0, posinf=1e6, neginf=0.0)


class MaxRatioDetector:
    """a_t = max_j z_tj / q_j, where q_j is the benign `q`-th percentile of feature j."""

    def __init__(self, q: float = 99.0):
        self.q = q
        self.scale = None

    def fit(self, Z_benign: np.ndarray) -> "MaxRatioDetector":
        self.scale = np.maximum(np.percentile(_clean(Z_benign), self.q, axis=0), 1e-9)
        return self

    def score(self, Z: np.ndarray) -> np.ndarray:
        return np.max(_clean(Z) / self.scale, axis=1)


def threshold_at_fpr(benign_scores: np.ndarray, fpr: float = 0.01) -> float:
    """Alarm threshold eta that gives the target false-positive rate on benign frames."""
    return float(np.quantile(benign_scores, 1.0 - fpr))


def m_of_n(above: np.ndarray, m: int = 2, n: int = 3) -> np.ndarray:
    """Alarm at t if at least m of the last n frames (including t) are above the threshold."""
    above = np.asarray(above, dtype=int)
    c = np.convolve(above, np.ones(n, dtype=int))[: len(above)]
    return c >= m


def trust(score: np.ndarray) -> np.ndarray:
    """theta_t = exp(-a_t)."""
    return np.exp(-np.maximum(np.asarray(score, dtype=float), 0.0))


def timestamp_check_score(stamps: np.ndarray, period: float = 0.1) -> np.ndarray:
    """B1: deviation of the header-time step from the nominal period; large if out of order.

    Under replay the attacker keeps the expected timestamps, so this score is the same
    as on benign data.
    """
    d = np.diff(np.asarray(stamps, dtype=float), prepend=np.nan)
    score = np.abs(d - period)
    score[d <= 0] = 1e3            # non-increasing time: strongest alarm
    score[0] = 0.0
    return score

In [ ]:
%%writefile {CODE}/freshscan/runinfo.py
"""Records the hardware and software used for each drive (needed for the run-time numbers, RQ4)."""

from __future__ import annotations

import json
import os
import platform
import subprocess
import time


def _cpu_model() -> str:
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def _gpu_name() -> str:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10)
        return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else "none"
    except Exception:
        return "none"


def collect(backend: str) -> dict:
    info = {"icp_backend": backend, "cpu": _cpu_model(), "cpu_count": os.cpu_count(),
            "gpu": _gpu_name(), "python": platform.python_version(),
            "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    try:
        import open3d
        info["open3d"] = open3d.__version__
    except Exception:
        info["open3d"] = "not installed"
    import numpy
    info["numpy"] = numpy.__version__
    return info


def write_run_info(cfg: dict, drive: str, backend: str) -> None:
    d = os.path.join(cfg["results_dir"], "run_info")
    os.makedirs(d, exist_ok=True)
    path = os.path.join(d, f"{drive}.json")
    if not os.path.exists(path):
        with open(path, "w") as fh:
            json.dump(collect(backend), fh, indent=2)

In [ ]:
%%writefile {CODE}/freshscan/checkpoint.py
"""Calibration fit ("checkpoint") and the scoring rule of FreshScan, shared by steps 2 and 5.

FreshScan has no neural network and no training. Everything it learns comes from benign
(normal) driving, in one calibration *fold*:
    * the real LiDAR scan period (median header step; one constant, never per frame, P1),
    * the benign 99th percentile of every check (detector scales, C5),
    * the alarm threshold eta (target false-positive rate on the fold's benign frames),
    * the seam-alarm threshold (much stricter, default 0.1%).

Evaluation protocol (config key `protocol`):
    leave_one_drive_out  each test drive is scored with a fold calibrated on the benign streams
                         of all *other* drives (no attack data is ever used);
    fixed                one fold calibrated on `calibration_drives` only.
An extra fold "all" is calibrated on every benign drive; use it for new data.

Alarm rule (C5): alarm at slot t if
    (a) the FreshScan score is above eta in at least m of the last n slots, or
    (b) the seam score (ICP fit features only) is above the seam threshold in slot t.
Rule (b) lets one clearly broken registration, which marks the start or end of a replay,
raise an alarm at once instead of being filtered out by the persistence rule.

Files (results/checkpoint/):
    freshscan_checkpoint.json   settings, protocol, and per fold: drives, scan period, scales, thresholds
    calibration_scores.npz      per fold: benign calibration scores of every method (to set new thresholds)
"""

from __future__ import annotations

import json
import os
import time

import numpy as np
import pandas as pd

from .detector import MaxRatioDetector, m_of_n, threshold_at_fpr, timestamp_check_score, trust
from .features import FEATURES, RESIDUAL_FEATURES, SEAM_FEATURES, motion_features

VERSION = 2
METHODS = {"B2: Seam only": list(SEAM_FEATURES), "B3: Residuals only": list(RESIDUAL_FEATURES),
           "FreshScan": list(FEATURES)}
B1 = "B1: Timestamp check"


def _key(m: str) -> str:
    return m.split(":")[0].replace(" ", "_")


class Fold:
    """One calibration fit and the scoring rule that uses it."""

    def __init__(self, name, fs, cal_drives, period, scales, thr, cal_scores, percentile=99.0):
        self.name, self.fs, self.cal_drives = name, fs, list(cal_drives)
        self.period = float(period)
        self.dets = {}
        for m, sc in scales.items():
            d = MaxRatioDetector(percentile); d.scale = np.asarray(sc, dtype=float); self.dets[m] = d
        self.thr = {k: float(v) for k, v in thr.items()}
        self.cal_scores = cal_scores          # method -> benign calibration scores

    # --- fitting ---
    @classmethod
    def fit(cls, name, fs, cal_streams: dict) -> "Fold":
        """cal_streams: {drive: raw benign feature DataFrame (slot 0 already removed)}."""
        lp = fs.get("lidar_period_s", "auto")
        if lp == "auto":
            steps = np.concatenate([np.diff(df.tau.values) for df in cal_streams.values()])
            lp = float(np.median(steps[steps > 0]))
        fold = cls(name, fs, list(cal_streams), lp, {}, {}, {}, fs["percentile"])
        feats = [fold.features(df) for df in cal_streams.values()]
        for m, cols in METHODS.items():
            Z = np.vstack([f[cols].values for f in feats])
            fold.dets[m] = MaxRatioDetector(fs["percentile"]).fit(Z)
            fold.cal_scores[m] = fold.dets[m].score(Z)
            fold.thr[m] = threshold_at_fpr(fold.cal_scores[m], fs["target_fpr"])
        fold.cal_scores[B1] = np.concatenate([timestamp_check_score(df.tau.values, fold.period)
                                              for df in cal_streams.values()])
        fold.thr[B1] = threshold_at_fpr(fold.cal_scores[B1], fs["target_fpr"])
        sf = fs.get("seam_alarm_fpr")
        fold.thr["seam_alarm"] = threshold_at_fpr(fold.cal_scores["B2: Seam only"], sf) if sf else float("inf")
        return fold

    # --- scoring ---
    def features(self, df: pd.DataFrame) -> pd.DataFrame:
        fs = self.fs
        df = df.copy()
        r_v, r_w, checked = motion_features(df.v_hat, df.w_hat, df.v_bar, df.w_bar,
                                            period_ratio=fs["period_s"] / self.period,
                                            k=int(fs.get("smooth_frames", 1)), v0=fs["v0"],
                                            v_min=fs.get("v_min", 0.0), w0=fs.get("w0"))
        df["r_v"], df["r_w"], df["checked"] = r_v, r_w, checked.astype(int)
        df["one_minus_fit"] = 1.0 - df["fit"].astype(float)
        return df

    def threshold(self, method: str, fpr: float | None = None) -> float:
        if fpr is None:
            return self.thr[method]
        return threshold_at_fpr(self.cal_scores[method], fpr)

    def score(self, df: pd.DataFrame, fpr: float | None = None, m: int | None = None,
              n: int | None = None, seam_fpr: float | None = None) -> pd.DataFrame:
        """Score one stream. Adds score::, thr::, alarm:: columns, `alarm` (FreshScan) and `trust`."""
        fs = self.fs
        m, n = m or fs["m"], n or fs["n"]
        df = self.features(df[df.slot > 0].reset_index(drop=True) if "slot" in df else df)
        for meth, cols in METHODS.items():
            df[f"score::{meth}"] = self.dets[meth].score(df[cols].values)
            df[f"thr::{meth}"] = self.threshold(meth, fpr)
        df[f"score::{B1}"] = timestamp_check_score(df.tau.values, self.period)
        df[f"thr::{B1}"] = self.threshold(B1, fpr)
        t_seam = self.thr["seam_alarm"] if seam_fpr is None else threshold_at_fpr(
            self.cal_scores["B2: Seam only"], seam_fpr)
        seam_alarm = df["score::B2: Seam only"].values > t_seam
        persist = {meth: m_of_n(df[f"score::{meth}"].values > df[f"thr::{meth}"].values, m, n)
                   for meth in METHODS}
        df["seam_alarm"] = seam_alarm.astype(int)
        df["alarm::B2: Seam only"] = seam_alarm.astype(int)                 # B2: seam rule only
        df["alarm::B3: Residuals only"] = persist["B3: Residuals only"].astype(int)
        df["alarm::FreshScan"] = (persist["FreshScan"] | seam_alarm).astype(int)
        df["alarm::B1: Timestamp check"] = m_of_n(df[f"score::{B1}"].values > df[f"thr::{B1}"].values,
                                                  m, n).astype(int)
        df["alarm"] = df["alarm::FreshScan"]
        df["trust"] = trust(df["score::FreshScan"].values)
        df["fold"] = self.name
        return df

    # --- (de)serialization ---
    def meta(self) -> dict:
        return {"calibration_drives": self.cal_drives, "lidar_period_s": self.period,
                "scales": {m: [float(x) for x in d.scale] for m, d in self.dets.items()},
                "thresholds": self.thr}


def save_checkpoint(path_dir: str, cfg: dict, folds: dict, protocol: str, run_info: dict | None = None) -> str:
    os.makedirs(path_dir, exist_ok=True)
    ck = {"freshscan_checkpoint_version": VERSION, "created": time.strftime("%Y-%m-%d %H:%M:%S"),
          "protocol": protocol, "freshscan": cfg["freshscan"], "replay": cfg["replay"],
          "calibration_drives": cfg["calibration_drives"], "test_drives": cfg["test_drives"],
          "methods": METHODS, "folds": {name: f.meta() for name, f in folds.items()},
          "run_info": run_info or {}}
    with open(os.path.join(path_dir, "freshscan_checkpoint.json"), "w") as fh:
        json.dump(ck, fh, indent=2)
    arrays = {f"{name}__{_key(m)}": np.asarray(s) for name, f in folds.items() for m, s in f.cal_scores.items()}
    np.savez_compressed(os.path.join(path_dir, "calibration_scores.npz"), **arrays)
    return path_dir


class Checkpoint:
    def __init__(self, path_dir: str):
        with open(os.path.join(path_dir, "freshscan_checkpoint.json")) as fh:
            self.meta = json.load(fh)
        cal = np.load(os.path.join(path_dir, "calibration_scores.npz"))
        fs = self.meta["freshscan"]
        self.folds = {}
        for name, fm in self.meta["folds"].items():
            scores = {m: cal[f"{name}__{_key(m)}"] for m in list(METHODS) + [B1]}
            self.folds[name] = Fold(name, fs, fm["calibration_drives"], fm["lidar_period_s"],
                                    fm["scales"], fm["thresholds"], scores, fs["percentile"])

    def fold_for(self, drive: str) -> Fold:
        return self.folds.get(drive, self.folds.get("all"))


def load_checkpoint(path_dir: str) -> Checkpoint:
    return Checkpoint(path_dir)

In [ ]:
%%writefile {CODE}/scripts/common.py
"""Shared helpers for the experiment scripts."""

from __future__ import annotations

import os
import sys

import yaml

HERE = os.path.dirname(os.path.abspath(__file__))
CODE = os.path.dirname(HERE)
sys.path.insert(0, CODE)


def load_config(path: str | None = None) -> dict:
    path = path or os.path.join(CODE, "configs", "default.yaml")
    with open(path) as fh:
        cfg = yaml.safe_load(fh)
    for key in ("kitti_root", "results_dir"):
        if not os.path.isabs(cfg[key]):
            cfg[key] = os.path.join(CODE, cfg[key])
    return cfg


def out_dir(cfg: dict, *parts: str) -> str:
    d = os.path.join(cfg["results_dir"], *parts)
    os.makedirs(d, exist_ok=True)
    return d

In [ ]:
%%writefile {CODE}/scripts/01_compute_features.py
"""Step 1: run FreshScan components C1-C3 on benign and replayed streams and save per-frame CSVs.

For every calibration drive: the benign stream.
For every test drive: the benign stream and one replayed stream per (replay age R, seed).

Output: results/features/<drive>__<setting>.csv with one row per slot:
    slot, sigma, label, seam, tau, v_bar, w_bar, v_bar_src, w_bar_src,
    v_hat, w_hat, fit, rmse, r_v, r_w, one_minus_fit, prep_ms, icp_ms, check_ms

Usage (from the repository root):
    python scripts/01_compute_features.py                     # all drives in the config
    python scripts/01_compute_features.py --drives 2011_09_30_drive_0018
"""

from __future__ import annotations

import argparse
import os
import time
from collections import OrderedDict

import numpy as np
import pandas as pd

from common import load_config, out_dir  # noqa: E402  (adds the repository root to sys.path)
from freshscan.features import feature_table
from freshscan.kitti_loader import load_drive
from freshscan.registration import icp, prepare, select_backend
from freshscan.runinfo import write_run_info
from freshscan.replay import ReplayConfig, inject


class FrameCache:
    """Keeps the most recently prepared frames (C1) in memory."""

    def __init__(self, drive, fs: dict, size: int = 512):
        self.drive, self.fs, self.size = drive, fs, size
        self.data: OrderedDict = OrderedDict()
        self.prep_ms: dict = {}

    def get(self, i: int) -> np.ndarray:
        if i in self.data:
            self.data.move_to_end(i)
            return self.data[i]
        t0 = time.perf_counter()
        pts = prepare(self.drive.frame(i), self.fs["r_min"], self.fs["r_max"], self.fs["voxel"])
        self.prep_ms[i] = 1e3 * (time.perf_counter() - t0)
        self.data[i] = pts
        if len(self.data) > self.size:
            self.data.popitem(last=False)
        return pts


def run_stream(drive, sigma, fs, frames: FrameCache, pair_cache: dict):
    """C2 on consecutive delivered frames. Returns per-slot LiDAR motion, fit and timings."""
    n, period = len(sigma), fs["period_s"]
    v_hat, w_hat = np.zeros(n), np.zeros(n)
    fit, rmse = np.ones(n), np.zeros(n)
    prep_ms, icp_ms = np.zeros(n), np.zeros(n)
    T_prev, f_prev, prev_key = np.eye(4), 1.0, None
    for t in range(n):
        frames.get(int(sigma[t]))
        prep_ms[t] = frames.prep_ms.get(int(sigma[t]), 0.0)
        if t == 0:
            continue
        a, b = int(sigma[t - 1]), int(sigma[t])
        key = (prev_key, a, b)
        if key not in pair_cache:
            init = T_prev if f_prev >= fs["fit_reset"] else np.eye(4)   # P2: LiDAR-only init
            t0 = time.perf_counter()
            res = icp(frames.get(b), frames.get(a), init=init, d_max=fs["d_max"], iters=fs["icp_iters"])
            pair_cache[key] = (res.forward, res.yaw, res.fitness, res.rmse,
                               1e3 * (time.perf_counter() - t0), res.T)
        fwd, yaw, f, e, ms, T = pair_cache[key]
        v_hat[t], w_hat[t] = fwd / period, yaw / period
        fit[t], rmse[t], icp_ms[t] = f, e, ms
        T_prev, f_prev, prev_key = T, f, (a, b)
    return v_hat, w_hat, fit, rmse, prep_ms, icp_ms


def slot_mean(x):
    """Average of the samples at frames t-1 and t (frame 0 keeps its own sample)."""
    x = np.asarray(x, dtype=float)
    return np.r_[x[:1], 0.5 * (x[1:] + x[:-1])]


def save_stream(path, drive, sigma, label, seam, out, fs):
    v_hat, w_hat, fit, rmse, prep_ms, icp_ms = out
    n = len(sigma)
    # Reference motion for slot t = mean of the IMU/GPS samples at both ends of the interval
    # [t-1, t] that ICP measures.
    v_bar, w_bar = slot_mean(drive.speed[:n]), slot_mean(drive.yaw_rate[:n])
    t0 = time.perf_counter()
    feats = feature_table(v_hat, w_hat, v_bar, w_bar, fit, rmse, fs["v0"])
    check_ms = 1e3 * (time.perf_counter() - t0) / max(n, 1)
    df = pd.DataFrame({
        "slot": np.arange(n), "sigma": sigma, "label": label, "seam": seam,
        "tau": drive.stamps[:n],                       # header time of slot t (kept by the attacker)
        "v_bar": v_bar, "w_bar": w_bar,
        "v_bar_src": slot_mean(drive.speed)[sigma], "w_bar_src": slot_mean(drive.yaw_rate)[sigma],  # motion when the frame was captured
        "v_hat": v_hat, "w_hat": w_hat, "fit": fit, "rmse": rmse,
        "r_v": feats["r_v"], "r_w": feats["r_w"], "one_minus_fit": feats["one_minus_fit"],
        "prep_ms": prep_ms, "icp_ms": icp_ms, "check_ms": check_ms,
    })
    df.to_csv(path, index=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--drives", nargs="*", default=None, help="subset of drives to process")
    args = ap.parse_args()
    cfg = load_config(args.config)
    fs, rp = cfg["freshscan"], cfg["replay"]
    feat_dir = out_dir(cfg, "features")
    backend = select_backend(fs.get("icp_backend", "auto"))
    print(f"ICP backend: {backend}")

    calib, test = cfg["calibration_drives"], cfg["test_drives"]
    todo = [d for d in calib + test if args.drives is None or d in args.drives]
    for name in todo:
        drive = load_drive(cfg["kitti_root"], name, fs["period_s"])
        n = len(drive)
        frames, pair_cache = FrameCache(drive, fs), {}
        streams = [("benign", np.arange(n), np.zeros(n, int), np.zeros(n, int))]
        if name in test:
            for R_s in rp["ages_s"]:
                for seed in rp["seeds"]:
                    rc = ReplayConfig(replay_s=R_s, win_min_s=rp["win_min_s"], win_max_s=rp["win_max_s"],
                                      gap_s=rp["gap_s"], period_s=fs["period_s"], seed=seed)
                    sigma, label, seam, R, wins = inject(n, rc)
                    if not wins:
                        print(f"[{name}] drive too short for R={R_s}s, skipped")
                        continue
                    streams.append((f"replay_R{R_s}_s{seed}", sigma, label, seam))
        for setting, sigma, label, seam in streams:
            path = os.path.join(feat_dir, f"{name}__{setting}.csv")
            if os.path.exists(path):
                print(f"[{name}] {setting:16s} exists, skipped"); continue
            t0 = time.time()
            out = run_stream(drive, sigma, fs, frames, pair_cache)
            save_stream(path, drive, sigma, label, seam, out, fs)
            write_run_info(cfg, name, backend)
            print(f"[{name}] {setting:16s} frames={n} attacked={int(label.sum())} "
                  f"({time.time() - t0:.0f}s) -> {os.path.relpath(path)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/scripts/02_evaluate.py
"""Step 2: calibrate FreshScan on benign drives, score every test stream, compute every result table.

Protocol (config `protocol`): leave_one_drive_out (default) calibrates each test drive on the benign
streams of all other drives; fixed calibrates once on `calibration_drives`. No attack data is used.
The calibration fits are saved as results/checkpoint/ (see freshscan/checkpoint.py).

Outputs (results/tables/):
    rq1_detection.csv    RQ1: AUROC and TPR@1% per method and replay age
    rq2_delay.csv        RQ2: detection delay, windows detected, false alarms per minute
    rq3_analysis.csv     RQ3: TPR@1% of FreshScan split by seam, speed difference, speed
    rq4_runtime.csv      RQ4: mean and p95 time per frame for each step
    dataset_summary.csv  numbers for the "Dataset" paragraph
    scores/<drive>__<setting>.csv  per-frame scores, alarms and trust (used for figures)

Usage (from the repository root):  python scripts/02_evaluate.py
"""

from __future__ import annotations

import argparse
import glob
import json
import os
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from common import load_config, out_dir  # noqa: E402
from freshscan.checkpoint import B1, METHODS, Fold, save_checkpoint


def load_streams(feat_dir: str) -> dict:
    out = {}
    for f in sorted(glob.glob(os.path.join(feat_dir, "*.csv"))):
        drive, setting = os.path.basename(f)[:-4].split("__")
        df = pd.read_csv(f)
        # slot 0 has no previous frame, so no LiDAR motion: leave it out of all statistics
        out[(drive, setting)] = df[df.slot > 0].reset_index(drop=True)
    if not out:
        raise FileNotFoundError(f"No feature CSVs in {feat_dir}. Run 01_compute_features.py first.")
    return out


def replay_age(setting: str):
    return int(setting.split("_")[1][1:]) if setting.startswith("replay") else None


def tpr_at(scores_att, thr):
    return float(np.mean(scores_att > thr)) if len(scores_att) else float("nan")


def auroc(ben, att):
    if len(ben) == 0 or len(att) == 0:
        return float("nan")
    y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
    return float(roc_auc_score(y, np.r_[ben, att]))


def window_stats(df, alarm):
    """Detection delay (frames from window start to first alarm) for each window."""
    lab = df.label.values
    starts = np.where(np.diff(np.r_[0, lab]) == 1)[0]
    ends = np.where(np.diff(np.r_[lab, 0]) == -1)[0]
    delays = []
    for s, e in zip(starts, ends):
        hit = np.where(alarm[s:e + 1])[0]
        delays.append(int(hit[0]) if len(hit) else None)
    return delays


def events(alarm) -> int:
    a = np.asarray(alarm, dtype=int)
    return int(((a == 1) & (np.r_[0, a[:-1]] == 0)).sum())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    fs = cfg["freshscan"]
    protocol = cfg.get("protocol", "leave_one_drive_out")
    streams = load_streams(out_dir(cfg, "features"))
    tab_dir, score_dir = out_dir(cfg, "tables"), out_dir(cfg, "tables", "scores")

    benign = {k[0]: streams[k] for k in streams if k[1] == "benign"}
    test_drives = [d for d in cfg["test_drives"] if d in benign]
    calib_drives = [d for d in cfg["calibration_drives"] if d in benign]
    if not test_drives or not calib_drives:
        raise RuntimeError("Need benign streams of calibration drives and of test drives.")

    # ---- calibration folds (benign data only) ----
    folds = {}
    for td in test_drives:
        cal = [d for d in calib_drives + test_drives if d != td] if protocol == "leave_one_drive_out" \
            else calib_drives
        folds[td] = Fold.fit(td, fs, {d: benign[d] for d in cal})
    folds["all"] = Fold.fit("all", fs, dict(benign))          # for new data (not used in the tables)
    for td in test_drives:
        f = folds[td]
        print(f"[{td}] calibrated on {len(f.cal_drives)} drives, scan period {f.period * 1e3:.2f} ms, "
              f"eta {f.thr['FreshScan']:.3f}, seam-alarm threshold {f.thr['seam_alarm']:.3f}")

    # ---- score every test stream with its fold ----
    scored = {}
    t0 = time.perf_counter(); n_scored = 0
    for k in streams:
        if k[0] not in test_drives:
            continue
        df = folds[k[0]].score(streams[k])
        scored[k] = df
        n_scored += len(df)
        df.to_csv(os.path.join(score_dir, f"{k[0]}__{k[1]}.csv"), index=False)
    score_ms = 1e3 * (time.perf_counter() - t0) / max(n_scored, 1)

    test = list(scored)
    benign_test = [k for k in test if k[1] == "benign"]
    ages = sorted({replay_age(k[1]) for k in test if k[1].startswith("replay")})
    all_methods = [B1] + list(METHODS)
    period = float(np.mean([folds[d].period for d in test_drives]))
    minutes = sum(len(scored[k]) for k in benign_test) * period / 60.0

    def pooled(method, keys, attacked):
        sc, hit = [], []
        for k in keys:
            d = scored[k]
            sel = (d.label == 1).values if attacked else np.ones(len(d), bool)
            sc.append(d.loc[sel, f"score::{method}"].values)
            hit.append(d.loc[sel, f"score::{method}"].values > d.loc[sel, f"thr::{method}"].values)
        return np.concatenate(sc), np.concatenate(hit)

    # ---- RQ1: frame-level detection ----
    rq1 = []
    for m in all_methods:
        ben, ben_hit = pooled(m, benign_test, False)
        for age in ages:
            att, att_hit = pooled(m, [k for k in test if replay_age(k[1]) == age], True)
            rq1.append(dict(method=m, replay_age_s=age, auroc=auroc(ben, att), tpr_at_1pct=float(att_hit.mean()),
                            benign_fpr_on_test=float(ben_hit.mean())))
    pd.DataFrame(rq1).to_csv(os.path.join(tab_dir, "rq1_detection.csv"), index=False)

    # ---- RQ2: windows, delay, false alarm events ----
    rq2 = []
    for m in list(METHODS):
        fa = sum(events(scored[k][f"alarm::{m}"].values) for k in benign_test)
        for age in ages:
            delays = []
            for k in test:
                if replay_age(k[1]) == age:
                    delays += window_stats(scored[k], scored[k][f"alarm::{m}"].values.astype(bool))
            hit = [d for d in delays if d is not None]
            rq2.append(dict(method=m, replay_age_s=age,
                            median_delay_frames=float(np.median(hit)) if hit else float("nan"),
                            p95_delay_frames=float(np.percentile(hit, 95)) if hit else float("nan"),
                            windows_detected_pct=100.0 * len(hit) / max(len(delays), 1),
                            n_windows=len(delays),
                            false_alarms_per_min=fa / max(minutes, 1e-9)))
    pd.DataFrame(rq2).to_csv(os.path.join(tab_dir, "rq2_delay.csv"), index=False)

    # ---- RQ3 (FreshScan, one replay age): frame-level TPR split as in the detectability analysis ----
    age = cfg["rq3"]["replay_age_s"]
    att = pd.concat([scored[k][scored[k].label == 1] for k in test if replay_age(k[1]) == age])
    hit = att["score::FreshScan"].values > att["thr::FreshScan"].values
    dv = np.abs(att.v_bar.values - att.v_bar_src.values)
    sp = np.abs(att.v_bar.values)
    e1, e2 = cfg["rq3"]["dv_bins"]; s1, s2 = cfg["rq3"]["speed_bins"]
    splits = [
        ("Position in window", "Seam (entry and exit)", att.seam.values == 1),
        ("Position in window", "Inside", att.seam.values == 0),
        ("|v(t)-v(t-R)|", f"<{e1} m/s", dv < e1),
        ("|v(t)-v(t-R)|", f"{e1}-{e2} m/s", (dv >= e1) & (dv <= e2)),
        ("|v(t)-v(t-R)|", f">{e2} m/s", dv > e2),
        ("Current speed", f"<{s1} m/s", sp < s1),
        ("Current speed", f"{s1}-{s2} m/s", (sp >= s1) & (sp <= s2)),
        ("Current speed", f">{s2} m/s", sp > s2),
    ]
    rq3 = [dict(split=a, bin=b, share_of_attacked_pct=100.0 * sel.mean(),
                tpr_at_1pct=float(hit[sel].mean()) if sel.any() else float("nan")) for a, b, sel in splits]
    pd.DataFrame(rq3).to_csv(os.path.join(tab_dir, "rq3_analysis.csv"), index=False)

    # ---- RQ4: run time ----
    allf = pd.concat([streams[k] for k in test])
    prep = allf.prep_ms[allf.prep_ms > 0]; icpm = allf.icp_ms[allf.icp_ms > 0]
    check = allf.check_ms + score_ms
    rq4 = [dict(step="Preparation (C1)", mean_ms=prep.mean(), p95_ms=prep.quantile(0.95)),
           dict(step="ICP (C2)", mean_ms=icpm.mean(), p95_ms=icpm.quantile(0.95)),
           dict(step="Checks and decision (C3, C5)", mean_ms=check.mean(), p95_ms=check.quantile(0.95))]
    tot = prep.mean() + icpm.mean() + check.mean()
    rq4.append(dict(step="Total", mean_ms=tot, p95_ms=prep.quantile(0.95) + icpm.quantile(0.95) + check.quantile(0.95)))
    pd.DataFrame(rq4).to_csv(os.path.join(tab_dir, "rq4_runtime.csv"), index=False)

    # ---- dataset summary ----
    def frames(keys): return int(sum(len(streams[k]) for k in keys))
    cal_frames = [sum(len(benign[d]) for d in folds[td].cal_drives) for td in test_drives]
    _, ben_hit = pooled("FreshScan", benign_test, False)
    summary = dict(protocol=protocol, calibration_drives=len(calib_drives),
                   drives_per_fold=int(np.mean([len(folds[td].cal_drives) for td in test_drives])),
                   calibration_frames_per_fold_mean=int(np.mean(cal_frames)),
                   test_drives=len(benign_test), test_frames=frames(benign_test),
                   test_minutes=round(minutes, 1), seeds=len(cfg["replay"]["seeds"]),
                   threshold_eta_mean=float(np.mean([folds[d].thr["FreshScan"] for d in test_drives])),
                   seam_threshold_mean=float(np.mean([folds[d].thr["seam_alarm"] for d in test_drives])),
                   lidar_period_ms=round(period * 1e3, 2),
                   benign_test_checked_pct=round(100.0 * np.mean(np.concatenate(
                       [scored[k].checked.values for k in benign_test])), 1),
                   benign_test_fpr_pct=round(100.0 * float(ben_hit.mean()), 2))
    pd.DataFrame([summary]).to_csv(os.path.join(tab_dir, "dataset_summary.csv"), index=False)
    with open(os.path.join(tab_dir, "thresholds.json"), "w") as fh:
        json.dump({name: f.thr for name, f in folds.items()}, fh, indent=2)
    print(f"Tables written to {tab_dir}")

    run_info = {}
    for f in sorted(glob.glob(os.path.join(cfg["results_dir"], "run_info", "*.json"))):
        run_info[os.path.basename(f)[:-5]] = json.load(open(f))
    ck = save_checkpoint(out_dir(cfg, "checkpoint"), cfg, folds, protocol, run_info)
    print(f"Checkpoint written to {ck}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/scripts/03_make_figures.py
"""Step 3: result and dataset figures (colour, vector PDF, 12.2 cm wide).

    results/figures/fig_roc.pdf            ROC curves for replay with R = 10 s (RQ1)
    results/figures/fig_features.pdf       each check on normal vs replayed frames (RQ1)
    results/figures/fig_replay_drive.pdf   one whole test drive under replay: every window and alarm (RQ2)
    results/figures/fig_timeseries.pdf     zoom on one replay window: speeds, ICP fitness, trust (RQ3)
    results/figures/fig_motion.pdf         LiDAR (ICP) motion vs IMU/GPS motion on normal driving (premise)
    results/figures/fig_dataset.pdf        speed / yaw-rate distributions and driven paths of all drives (setup)
    results/figures/fig_bev.pdf            bird's-eye view: fresh scan pair vs a replay seam (framework)
    results/tables/dataset_drives.csv      one row per drive (setup table)
    results/tables/summary_stats.json        other numbers (correlations, errors, seam overlap)

Colours: categorical slots in a fixed order (validated for colour-vision deficiency), and every series
also has its own line style or marker so the figures stay readable in print.

Usage (from the repository root):  python scripts/03_make_figures.py [--drive DRIVE]
"""

from __future__ import annotations

import argparse
import glob
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
from sklearn.metrics import roc_curve  # noqa: E402

from common import load_config, out_dir  # noqa: E402

# --- style ---
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"   # categorical slots 1-4
CRIT = "#d03b3b"                                                          # status: alarm
INK, INK2, MUTED, SHADE = "#0b0b0b", "#52514e", "#8a8985", "#ececea"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
TEXTW = 4.8                                                               # figure width [in] (12.2 cm)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Termes", "Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 7.5, "axes.titlesize": 7.5, "axes.labelsize": 7.5,
    "legend.fontsize": 6.8, "xtick.labelsize": 6.8, "ytick.labelsize": 6.8,
    "axes.linewidth": 0.6, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "text.color": INK, "axes.grid": True, "grid.color": "#e3e2de",
    "grid.linewidth": 0.5, "axes.axisbelow": True, "lines.linewidth": 1.2, "pdf.fonttype": 42,
    "legend.frameon": False, "axes.spines.top": False, "axes.spines.right": False,
})
METHODS = [  # entity -> (colour, line style, width); fixed for every figure
    ("FreshScan", BLUE, "-", 1.6),
    ("B3: Residuals only", ORANGE, "-.", 1.2),
    ("B2: Seam only", AQUA, "--", 1.2),
    ("B1: Timestamp check", YELLOW, ":", 1.4),
]


def save(fig, fig_dir, name):
    fig.savefig(os.path.join(fig_dir, f"{name}.pdf"))
    fig.savefig(os.path.join(fig_dir, f"{name}.png"), dpi=200)
    plt.close(fig)
    print("  wrote", name)


def shade_windows(ax, t, lab):
    starts = np.where(np.diff(np.r_[0, lab]) == 1)[0]
    ends = np.where(np.diff(np.r_[lab, 0]) == -1)[0]
    for s, e in zip(starts, ends):
        ax.axvspan(t[s], t[e], color=SHADE, lw=0, zorder=0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--drive", default=None, help="test drive for the replay figures")
    args = ap.parse_args()
    cfg = load_config(args.config)
    fs = cfg["freshscan"]
    res = cfg["results_dir"]
    score_dir = os.path.join(res, "tables", "scores")
    fig_dir, tab_dir = out_dir(cfg, "figures"), out_dir(cfg, "tables")
    age = cfg["rq3"]["replay_age_s"]
    files = sorted(glob.glob(os.path.join(score_dir, "*.csv")))
    if not files:
        raise FileNotFoundError("Run 02_evaluate.py first.")
    dfs = {os.path.basename(f)[:-4]: pd.read_csv(f) for f in files}
    ck_path = os.path.join(res, "checkpoint", "freshscan_checkpoint.json")
    ck = json.load(open(ck_path)) if os.path.exists(ck_path) else {"folds": {}}
    folds = ck.get("folds", {})
    period_all = folds.get("all", {}).get("lidar_period_s", fs["period_s"])
    stats = {}

    def period_of(df):
        return folds.get(str(df.fold.iloc[0]), {}).get("lidar_period_s", period_all) if "fold" in df else period_all

    # --- Fig: ROC (RQ1) ---
    ben = pd.concat([d for k, d in dfs.items() if k.endswith("__benign")])
    att = pd.concat([d[d.label == 1] for k, d in dfs.items() if f"__replay_R{age}_" in k])
    fig, ax = plt.subplots(figsize=(2.35, 2.05))
    ax.plot([0, 1], [0, 1], color=MUTED, lw=0.6, ls=(0, (2, 2)))
    for m, c, ls, lw in METHODS[::-1]:
        col = f"score::{m}"
        fpr, tpr, _ = roc_curve(np.r_[np.zeros(len(ben)), np.ones(len(att))], np.r_[ben[col], att[col]])
        ax.plot(fpr, tpr, color=c, ls=ls, lw=lw, label=m.replace("FreshScan", "FreshScan (ours)"))
    ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], loc="lower right", handlelength=2.4, frameon=True, framealpha=0.95, edgecolor="#c9c8c3", facecolor="white")
    fig.tight_layout(pad=0.3)
    save(fig, fig_dir, "fig_roc")

    # --- Fig: feature distributions, normal vs replay (RQ1) ---
    att_in = att[att.seam == 0]
    att_seam = att[att.seam == 1]
    panels = [("r_v", "Speed residual $r^v_t$"), ("r_w", "Yaw-rate residual $r^\\omega_t$"),
              ("one_minus_fit", "ICP misfit $1-f_t$"), ("score::FreshScan", "FreshScan score $a_t$")]
    fig, axs = plt.subplots(1, 4, figsize=(TEXTW, 1.75))
    for ax, (c, lab) in zip(axs, panels):
        logx = c.startswith("score")
        allv = np.r_[ben[c], att[c]]
        if logx:
            b = np.logspace(np.log10(max(np.percentile(allv, 0.5), 1e-3)), np.log10(np.percentile(allv, 99.9)), 45)
        else:
            b = np.linspace(0, np.percentile(allv, 99.5), 45)
        clip = lambda v: np.clip(v, b[0], b[-1])  # noqa: E731
        ax.hist(clip(ben[c]), bins=b, density=True, color=BLUE, alpha=0.35, lw=0, label="normal")
        ax.hist(clip(ben[c]), bins=b, density=True, histtype="step", color=BLUE, lw=0.9)
        ax.hist(clip(att_in[c]), bins=b, density=True, histtype="step", color=ORANGE, lw=1.1, label="replayed, inside")
        ax.hist(clip(att_seam[c]), bins=b, density=True, histtype="step", color=INK, lw=0.8, ls=(0, (2, 1.2)),
                label="replayed, seam")
        ax.set_yscale("log"); ax.set_xlabel(lab); ax.tick_params(axis="y", labelsize=6)
        if logx:
            ax.set_xscale("log")
            thr = float(np.median(pd.concat(dfs.values())["thr::FreshScan"]))
            ax.axvline(thr, color=CRIT, lw=0.9, ls="--")
    axs[0].set_ylabel("Density (log)")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, ["normal frames", "replayed frames inside a window", "replayed frames at a seam"],
               loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0), handlelength=2.0)
    fig.tight_layout(pad=0.3, w_pad=0.6, rect=(0, 0, 1, 0.88))
    save(fig, fig_dir, "fig_features")

    # --- Fig: whole drive under replay (RQ2) ---
    keys = sorted(k for k in dfs if f"__replay_R{age}_s0" in k)
    if args.drive:
        keys = [k for k in keys if k.startswith(args.drive)] or keys
    df = dfs[keys[0]]
    p = period_of(df)
    t = df.slot.values * p
    lab = df.label.values
    thr = float(df["thr::FreshScan"].iloc[0])
    fig, axs = plt.subplots(3, 1, figsize=(TEXTW, 3.55), sharex=True,
                            gridspec_kw={"height_ratios": [1, 1.1, 1.25]})
    for ax in axs:
        shade_windows(ax, t, lab)
    axs[0].plot(t, df.slot, color=MUTED, lw=0.9, ls=(0, (3, 2)), label="expected frame (fresh)")
    axs[0].plot(t, df.sigma, color=BLUE, lw=1.0, label="delivered frame")
    axs[0].set_ylabel("Frame index")
    axs[1].plot(t, df.v_bar, color=INK, lw=0.9, label="IMU/GPS speed now")
    axs[1].plot(t, df.v_bar_src, color=ORANGE, lw=0.9, ls="--", label="speed when the delivered frame was captured")
    axs[1].set_ylabel("Speed (m/s)")
    axs[2].plot(t, df["score::FreshScan"], color=BLUE, lw=0.8, label="FreshScan score")
    axs[2].axhline(thr, color=INK2, lw=0.8, ls="--", label="alarm threshold")
    al = df.alarm.values == 1
    sa = df.seam_alarm.values == 1
    ymax = np.nanmax(df["score::FreshScan"]) * 3
    axs[2].plot(t[al], np.full(al.sum(), ymax * 0.55), "|", color=CRIT, ms=5, mew=1.0, label="alarm")
    axs[2].plot(t[sa], np.full(sa.sum(), ymax), "v", color=CRIT, ms=3.2, mfc="white", mew=0.8, label="seam alarm")
    axs[2].set_yscale("log"); axs[2].set_ylabel("Score (log)"); axs[2].set_xlabel("Time (s)")

    axs[2].set_ylim(np.nanmin(df["score::FreshScan"]) / 2.5, ymax * 2.2)
    axs[2].set_xlim(t[0], t[-1])
    for ax in axs:
        ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.03), ncol=4, borderaxespad=0.1, columnspacing=1.2)
    fig.tight_layout(pad=0.3, h_pad=1.5)
    save(fig, fig_dir, "fig_replay_drive")
    nw = int((np.diff(np.r_[0, lab]) == 1).sum())
    stats["replay_drive"] = {"stream": keys[0], "minutes": round(len(df) * p / 60, 2), "windows": nw}

    # --- Fig: zoom on one replay window (RQ3) ---
    starts = np.where(np.diff(np.r_[0, lab]) == 1)[0]
    ends = np.where(np.diff(np.r_[lab, 0]) == -1)[0]
    s0, e0 = starts[0], ends[0]
    lo, hi = max(0, s0 - 150), min(len(df), e0 + 150)
    tt = t[lo:hi]
    ratio = fs["period_s"] / p
    fig, axs = plt.subplots(3, 1, figsize=(TEXTW * 0.82, 2.55), sharex=True)
    for ax in axs:
        ax.axvspan(t[s0], t[e0], color=SHADE, lw=0, zorder=0)
    axs[0].plot(tt, df.v_bar.values[lo:hi], color=INK, lw=1.0, label=r"IMU/GPS speed $\bar v_t$")
    axs[0].plot(tt, df.v_hat.values[lo:hi] * ratio, color=BLUE, lw=1.0, ls="--", label=r"LiDAR speed $\hat v_t$")
    axs[0].set_ylabel("Speed (m/s)")
    axs[0].legend(loc="lower center", bbox_to_anchor=(0.5, 0.97), ncol=2, borderaxespad=0)
    axs[1].plot(tt, df.fit.values[lo:hi], color=AQUA, lw=1.0)
    axs[1].set_ylabel(r"ICP fitness $f_t$")
    axs[2].plot(tt, df.trust.values[lo:hi], color=BLUE, lw=1.0, label=r"trust $\theta_t$")
    axs[2].axhline(np.exp(-thr), color=INK2, lw=0.8, ls="--", label="alarm threshold")
    a2 = df.alarm.values[lo:hi] == 1
    axs[2].plot(tt[a2], np.full(a2.sum(), 0.02), "|", color=CRIT, ms=4, mew=1.0, label="alarm")
    axs[2].set_ylabel(r"Trust $\theta_t$"); axs[2].set_xlabel("Time (s)")
    axs[2].legend(loc="lower right", fontsize=6, ncol=1, borderaxespad=0.2, frameon=True, framealpha=0.95, edgecolor="#c9c8c3")
    axs[2].set_xlim(tt[0], tt[-1])
    fig.tight_layout(pad=0.3, h_pad=0.2)
    save(fig, fig_dir, "fig_timeseries")

    # --- normal (benign) streams of every drive, for the next figures ---
    feats = {}
    for f in sorted(glob.glob(os.path.join(res, "features", "*__benign.csv"))):
        name = os.path.basename(f).split("__")[0]
        d = pd.read_csv(f)
        feats[name] = d[d.slot > 0].reset_index(drop=True)
    if feats:
        role = lambda n: "test" if n in cfg["test_drives"] else "normal only"  # noqa: E731
        v_min = fs.get("v_min", 0.0)
        rows = []
        for n, d in feats.items():
            rows.append(dict(drive=n, role=role(n), frames=len(d) + 1, minutes=round((len(d) + 1) * period_all / 60, 2),
                             distance_km=round(float(np.sum(np.abs(d.v_bar)) * period_all / 1000), 2),
                             mean_speed_mps=round(float(d.v_bar.mean()), 1), max_speed_mps=round(float(d.v_bar.max()), 1),
                             below_vmin_pct=round(100 * float(np.mean(np.abs(d.v_bar) < v_min)), 1),
                             mean_abs_yaw_deg_s=round(float(np.degrees(np.abs(d.w_bar)).mean()), 1),
                             icp_fit_median=round(float(d.fit.median()), 3)))
        pd.DataFrame(rows).to_csv(os.path.join(tab_dir, "dataset_drives.csv"), index=False)

        # --- Fig: motion agreement (premise) ---
        allb = pd.concat(feats.values())
        vh, wh = allb.v_hat * fs["period_s"] / period_all, allb.w_hat * fs["period_s"] / period_all
        moving = np.abs(allb.v_bar) >= v_min
        ev, ew = (vh - allb.v_bar)[moving], np.degrees(wh - allb.w_bar)[moving]
        stats["motion"] = {
            "frames": int(len(allb)), "frames_moving": int(moving.sum()),
            "corr_speed": round(float(np.corrcoef(allb.v_bar, vh)[0, 1]), 4),
            "corr_yaw": round(float(np.corrcoef(allb.w_bar, wh)[0, 1]), 4),
            "speed_err_median_abs_mps": round(float(np.median(np.abs(ev))), 3),
            "speed_err_p95_abs_mps": round(float(np.percentile(np.abs(ev), 95)), 3),
            "yaw_err_median_abs_deg_s": round(float(np.median(np.abs(ew))), 3),
            "yaw_err_p95_abs_deg_s": round(float(np.percentile(np.abs(ew), 95)), 3)}
        fig, axs = plt.subplots(1, 2, figsize=(TEXTW * 0.92, 2.25), layout="constrained")
        for ax, x, y, lab, unit, cc in [
                (axs[0], allb.v_bar, vh, "speed", "m/s", stats["motion"]["corr_speed"]),
                (axs[1], np.degrees(allb.w_bar), np.degrees(wh), "yaw rate", "deg/s", stats["motion"]["corr_yaw"])]:
            hb = ax.hexbin(x, y, gridsize=55, bins="log", cmap=BLUES, mincnt=1, linewidths=0)
            lim = [min(x.min(), y.min()), max(x.max(), y.max())]
            ax.plot(lim, lim, color=INK2, lw=0.7, ls=(0, (3, 2)))
            ax.set_xlabel(f"IMU/GPS {lab} ({unit})"); ax.set_ylabel(f"LiDAR (ICP) {lab} ({unit})")
            ax.text(0.04, 0.95, f"correlation {cc:.3f}", transform=ax.transAxes, va="top", fontsize=6.8)
            ax.set_aspect("equal", adjustable="box")
        cb = fig.colorbar(hb, ax=axs, fraction=0.04, pad=0.01, shrink=0.9)
        cb.set_label("Frames (log)", fontsize=6.5); cb.ax.tick_params(labelsize=6)
        cb.outline.set_linewidth(0.4)
        save(fig, fig_dir, "fig_motion")

        # --- Fig: dataset (speed / yaw distributions, driven paths) ---
        names = sorted(feats, key=lambda n: (role(n) != "test", n))
        fig = plt.figure(figsize=(TEXTW, 3.05))
        top = fig.add_gridspec(1, 2, left=0.10, right=0.985, top=0.895, bottom=0.585, wspace=0.32)
        bot = fig.add_gridspec(1, len(names), left=0.02, right=0.985, top=0.36, bottom=0.085, wspace=0.12)
        ax1, ax2 = fig.add_subplot(top[0, 0]), fig.add_subplot(top[0, 1])
        groups = [("test drives (replay injected)", [n for n in names if role(n) == "test"], ORANGE, "--"),
                  ("normal-only drives", [n for n in names if role(n) != "test"], BLUE, "-")]
        vb = np.linspace(0, 28, 57)
        wb = np.linspace(0, 40, 41)
        for lab_, ns, c, ls in groups:
            v = np.concatenate([feats[n].v_bar.values for n in ns])
            w = np.degrees(np.abs(np.concatenate([feats[n].w_bar.values for n in ns])))
            ax1.hist(np.clip(v, 0, 28), bins=vb, density=True, histtype="step", color=c, ls=ls, lw=1.2, label=lab_)
            ax2.hist(np.clip(w, 0, 40), bins=wb, density=True, histtype="step", color=c, ls=ls, lw=1.2, label=lab_)
        ax1.axvline(v_min, color=INK2, lw=0.7, ls=":")
        ym = ax1.get_ylim()[1]
        ax1.set_ylim(0, ym * 1.28)
        ax1.annotate("motion checks abstain below 2 m/s", xy=(v_min, ym * 1.17), xytext=(4.5, ym * 1.17),
                     fontsize=6, color=INK2, va="center", arrowprops=dict(arrowstyle="-", color=INK2, lw=0.5))
        ax1.set_xlabel("IMU/GPS speed (m/s)"); ax1.set_ylabel("Density")
        ax2.set_xlabel("|Yaw rate| (deg/s)"); ax2.set_yscale("log"); ax2.set_ylabel("Density (log)")
        for ax, cap in [(ax1, "(a) Speed"), (ax2, "(b) Turning")]:
            ax.text(0.5, -0.43, cap, transform=ax.transAxes, ha="center", va="top", fontsize=7.5)
        h, l = ax1.get_legend_handles_labels()
        fig.legend(h, l, loc="upper center", ncol=2, bbox_to_anchor=(0.55, 1.0))
        fig.text(0.5, 0.045, "(c) Driven paths (integrated from IMU/GPS; dot = start)", ha="center", va="top", fontsize=7.5)
        for i, n in enumerate(names):
            ax = fig.add_subplot(bot[0, i])
            d = feats[n]
            yaw = np.cumsum(d.w_bar.values) * period_all
            x = np.cumsum(d.v_bar.values * np.cos(yaw)) * period_all
            y = np.cumsum(d.v_bar.values * np.sin(yaw)) * period_all
            c = ORANGE if role(n) == "test" else BLUE
            ax.plot(x, y, color=c, lw=0.9, ls="-")
            ax.plot(x[:1], y[:1], "o", ms=2.6, color=INK)
            ax.set_aspect("equal", "datalim")
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for sp in ax.spines.values():
                sp.set_visible(True); sp.set_color("#c9c8c3"); sp.set_linewidth(0.5)
            short = n.replace("2011_", "").replace("_drive_", "/")
            ax.set_title(f"{short}, {np.sum(np.abs(d.v_bar)) * period_all / 1000:.1f} km", fontsize=5.6, pad=2)
        save(fig, fig_dir, "fig_dataset")

    # --- Fig: bird's-eye view of a replay seam (framework) ---
    samples = sorted(glob.glob(os.path.join(res, "dataset_samples", "*.npz")))
    if samples:
        from scipy.spatial import cKDTree

        def overlap(cur, other, d=None):
            dist, _ = cKDTree(other).query(cur, distance_upper_bound=d or fs["d_max"])
            return float(np.isfinite(dist).mean())
        scored_s = []
        for f in samples:
            z = np.load(f)
            scored_s.append((overlap(z["cur"], z["prev"]) - overlap(z["cur"], z["old"]), f))
        pick = [f for f in samples if args.drive and os.path.basename(f).startswith(args.drive)] or [max(scored_s)[1]]
        z = np.load(pick[0]); name = os.path.basename(pick[0])[:-4]
        of, oo = overlap(z["cur"], z["prev"]), overlap(z["cur"], z["old"])
        fig, axs = plt.subplots(1, 3, figsize=(TEXTW, 1.95))
        b = z["bev"]
        o = np.argsort(b[:, 2])
        axs[0].scatter(b[o, 0], b[o, 1], s=0.05, c=b[o, 2], cmap=BLUES, vmin=-3.5, vmax=1.0, lw=0, rasterized=True)
        axs[0].set_title("(a) One scan (color: height)")
        for ax, other, lab_, ttl, ov in [(axs[1], z["prev"], "previous scan", "(b) Fresh pair", of),
                                          (axs[2], z["old"], "scan from 10 s earlier", "(c) Replay seam", oo)]:
            ax.scatter(z["cur"][:, 0], z["cur"][:, 1], s=0.15, color=BLUE, lw=0, rasterized=True, label="current scan")
            ax.scatter(other[:, 0], other[:, 1], s=0.15, color=ORANGE, lw=0, rasterized=True, label=lab_)
            ax.set_title(ttl)
            ax.text(0.03, 0.03, f"{100 * ov:.0f}% of points have\na match within {fs['d_max']:g} m", transform=ax.transAxes,
                    fontsize=6, va="bottom", bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#c9c8c3", lw=0.4, alpha=0.92))
            ax.legend(loc="upper left", markerscale=10, handletextpad=0.1, borderaxespad=0.15, fontsize=5.8,
                      frameon=True, framealpha=0.92, edgecolor="#c9c8c3")
        for ax in axs:
            ax.set_aspect("equal"); ax.set_xlim(-35, 35); ax.set_ylim(-35, 35)
            ax.set_xticks([-30, 0, 30]); ax.set_yticks([-30, 0, 30]); ax.grid(False)
            ax.set_xlabel("x (m), driving direction")
        axs[0].set_ylabel("y (m)")
        fig.tight_layout(pad=0.3, w_pad=0.4)
        save(fig, fig_dir, "fig_bev")
        stats["bev"] = {"drive": name, "frame": int(z["index"]), "speed_mps": round(float(z["speed"]), 1),
                        "overlap_fresh_pct": round(100 * of, 1), "overlap_10s_pct": round(100 * oo, 1)}

    with open(os.path.join(tab_dir, "summary_stats.json"), "w") as fh:
        json.dump(stats, fh, indent=2)
    print(json.dumps(stats, indent=2))

    print(f"Figures written to {fig_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/scripts/04_visualize_dataset.py
"""Step 4: dataset and behaviour figures.

Two modes:
  1. --save-samples DRIVE ...   (needs the raw KITTI scans; the notebook calls it right after step 1
     of each drive, before the scans are deleted) saves a few scans of each drive to
     results/dataset_samples/<drive>.npz so the bird's-eye-view figure can be drawn later.
  2. default: makes all figures from the feature CSVs, scores and checkpoint (no raw scans needed):
       fig_ds_overview          speed and yaw rate of every drive over time (normal-only vs test)
       fig_ds_distributions     speed / yaw-rate histograms, share of time below the abstain speed
       fig_ds_trajectories      driven path of every drive (integrated from IMU/GPS speed and yaw rate)
       fig_ds_motion_agreement  LiDAR (ICP) motion vs IMU/GPS motion on benign frames
       fig_ds_features          benign vs replayed distributions of every feature and the final score
       fig_ds_replay_example    one replay stream: delivered frame index, speeds, score and alarms
       fig_ds_bev               bird's-eye view: a fresh scan pair vs a replayed (10 s old) pair
     and tables/dataset_drives.csv (one row per drive: frames, minutes, km, speeds, abstain share, ICP fit).

Figures go to results/figures/dataset/ as PDF and PNG.

Usage (from the repository root):
    python scripts/04_visualize_dataset.py
    python scripts/04_visualize_dataset.py --save-samples 2011_09_30_drive_0018
"""

from __future__ import annotations

import argparse
import glob
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from common import load_config, out_dir  # noqa: E402

plt.rcParams.update({"font.size": 8, "axes.linewidth": 0.6, "lines.linewidth": 0.9,
                     "legend.fontsize": 7, "pdf.fonttype": 42, "axes.grid": True, "grid.alpha": 0.3})


SCALE = 1.0          # --hires: larger text for high-resolution PNGs
DPI = 180


def fsz(x):
    return x * SCALE


def save(fig, d, name, layout=True):
    if layout:
        fig.tight_layout()
    for ext in (("png",) if DPI > 200 else ("pdf", "png")):
        fig.savefig(os.path.join(d, f"{name}.{ext}"), dpi=DPI)
    plt.close(fig)
    print("  wrote", name)


# ----------------------------------------------------------------------------------------------
def save_samples(cfg, drives):
    from freshscan.kitti_loader import load_drive
    from freshscan.registration import prepare
    fs = cfg["freshscan"]
    d = out_dir(cfg, "dataset_samples")
    lag = int(round(10.0 / fs["period_s"]))                 # "replayed" scan: 10 s earlier
    for name in drives:
        drv = load_drive(cfg["kitti_root"], name, fs["period_s"])
        sp = np.asarray(drv.speed)
        cand = np.where((sp > 5.0) & (np.arange(len(sp)) > lag + 1))[0]
        i = int(cand[len(cand) // 2]) if len(cand) else len(sp) // 2
        i = max(i, lag + 1)
        np.savez_compressed(os.path.join(d, f"{name}.npz"),
                            index=i, lag=lag, speed=sp[i], speed_old=sp[i - lag],
                            bev=prepare(drv.frame(i), fs["r_min"], fs["r_max"], 0.2).astype(np.float32),
                            cur=prepare(drv.frame(i), fs["r_min"], fs["r_max"], fs["voxel"]).astype(np.float32),
                            prev=prepare(drv.frame(i - 1), fs["r_min"], fs["r_max"], fs["voxel"]).astype(np.float32),
                            old=prepare(drv.frame(i - lag), fs["r_min"], fs["r_max"], fs["voxel"]).astype(np.float32))
        print(f"[{name}] saved sample scans (frame {i}, speed {sp[i]:.1f} m/s)")


# ----------------------------------------------------------------------------------------------
def load_benign(cfg):
    out = {}
    for f in sorted(glob.glob(os.path.join(cfg["results_dir"], "features", "*__benign.csv"))):
        name = os.path.basename(f).split("__")[0]
        df = pd.read_csv(f)
        out[name] = df[df.slot > 0].reset_index(drop=True)
    return out


def role(cfg, name):
    return "normal-only" if name in cfg["calibration_drives"] else "test"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--save-samples", nargs="+", default=None, metavar="DRIVE")
    ap.add_argument("--hires", action="store_true",
                    help="high-resolution PNGs (800 dpi, larger text)")
    args = ap.parse_args()
    cfg = load_config(args.config)
    global SCALE, DPI
    if args.hires:
        SCALE, DPI = 1.55, 800
        plt.rcParams.update({"font.size": fsz(8), "legend.fontsize": fsz(7), "xtick.labelsize": fsz(7),
                             "ytick.labelsize": fsz(7), "axes.titlesize": fsz(8), "axes.labelsize": fsz(8)})
    if args.save_samples:
        save_samples(cfg, args.save_samples)
        return

    fs = cfg["freshscan"]
    figd = out_dir(cfg, "figures", "hires_png" if args.hires else "dataset")
    ck_path = os.path.join(cfg["results_dir"], "checkpoint", "freshscan_checkpoint.json")
    ck = json.load(open(ck_path)) if os.path.exists(ck_path) else None
    period = ck["folds"]["all"]["lidar_period_s"] if ck and "folds" in ck else fs["period_s"]
    ratio = fs["period_s"] / period
    v_min = fs.get("v_min", 0.0)
    benign = load_benign(cfg)
    if not benign:
        raise FileNotFoundError("No benign feature CSVs. Run step 1 first.")
    names = list(benign)
    print(f"Dataset figures for {len(names)} drives -> {figd}")

    # ---- table: one row per drive ----
    rows = []
    for n in names:
        d = benign[n]
        rows.append(dict(drive=n, role=role(cfg, n), frames=len(d) + 1,
                         minutes=round((len(d) + 1) * period / 60, 2),
                         distance_km=round(float(np.sum(np.abs(d.v_bar)) * period / 1000), 2),
                         mean_speed_mps=round(float(d.v_bar.mean()), 2), max_speed_mps=round(float(d.v_bar.max()), 2),
                         below_vmin_pct=round(100 * float(np.mean(np.abs(d.v_bar) < v_min)), 1),
                         mean_abs_yaw_deg_s=round(float(np.degrees(np.abs(d.w_bar)).mean()), 2),
                         icp_fit_median=round(float(d.fit.median()), 3), icp_fit_min=round(float(d.fit.min()), 3)))
    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(out_dir(cfg, "tables"), "dataset_drives.csv"), index=False)
    print(tab.to_string(index=False))

    # ---- 1. overview ----
    fig, axes = plt.subplots(len(names), 1, figsize=(7.2, (1.05 if SCALE > 1 else 1.25) * len(names) + 0.4), sharex=False, squeeze=False)
    for ax, n in zip(axes[:, 0], names):
        d = benign[n]; t = d.slot.values * period
        ax.plot(t, d.v_bar, color="black", label="speed [m/s]")
        ax2 = ax.twinx(); ax2.plot(t, np.degrees(d.w_bar), color="tab:blue", lw=0.6, label="yaw rate [deg/s]")
        ax2.set_ylabel("deg/s", fontsize=fsz(7)); ax2.grid(False)
        ax.axhline(v_min, color="tab:red", ls=":", lw=0.7)
        ax.set_ylabel("m/s"); ax.set_title(f"{n} ({role(cfg, n)})", fontsize=fsz(8), loc="left")
    axes[-1, 0].set_xlabel("time [s]")
    save(fig, figd, "fig_ds_overview")

    # ---- 2. distributions ----
    cal = pd.concat([benign[n] for n in names if role(cfg, n) == "normal-only"] or [pd.DataFrame(columns=["v_bar", "w_bar"])])
    tst = pd.concat([benign[n] for n in names if role(cfg, n) == "test"] or [pd.DataFrame(columns=["v_bar", "w_bar"])])
    fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.6 if SCALE > 1 else 2.2))
    bins = np.linspace(-1, max(30, float(pd.concat([cal.v_bar, tst.v_bar]).max()) + 1), 40)
    for df, lab, st in [(cal, "normal-only", "step"), (tst, "test", "stepfilled")]:
        if len(df):
            ax[0].hist(df.v_bar, bins=bins, density=True, histtype=st, alpha=0.5 if st == "stepfilled" else 1, label=lab)
            ax[1].hist(np.degrees(np.abs(df.w_bar)), bins=40, density=True, histtype=st,
                       alpha=0.5 if st == "stepfilled" else 1, label=lab, log=True)
    ax[0].axvline(v_min, color="tab:red", ls=":"); ax[0].set_xlabel("speed [m/s]")
    (ax[1] if SCALE > 1 else ax[0]).legend(loc="upper right")
    ax[1].set_xlabel("|yaw rate| [deg/s]")
    ax[2].barh(tab.drive.str.replace("2011_", "").str.replace("_drive_", " / "), tab.below_vmin_pct,
               color=["0.3" if r == "normal-only" else "0.7" for r in tab.role])
    ax[2].set_xlabel(f"% time below {v_min:g} m/s"); ax[2].tick_params(axis="y", labelsize=fsz(6))
    save(fig, figd, "fig_ds_distributions")

    # ---- 3. trajectories (dead reckoning from IMU/GPS speed and yaw rate) ----
    if SCALE > 1:
        ncol = (len(names) + 1) // 2
        fig, axes = plt.subplots(2, ncol, figsize=(7.2, 4.4), squeeze=False)
        for extra in axes.ravel()[len(names):]:
            extra.axis("off")
        axes = axes.ravel()[None, :len(names)]
    else:
        fig, axes = plt.subplots(1, len(names), figsize=(1.6 * len(names), 2.0), squeeze=False)
    for ax, n in zip(axes[0], names):
        d = benign[n]
        yaw = np.cumsum(d.w_bar.values) * period
        x = np.cumsum(d.v_bar.values * np.cos(yaw)) * period
        y = np.cumsum(d.v_bar.values * np.sin(yaw)) * period
        ax.plot(x, y, color="black" if role(cfg, n) == "normal-only" else "tab:blue")
        ax.plot(x[:1], y[:1], "o", ms=3, color="tab:green")
        ax.set_aspect("equal", "datalim")
        if SCALE > 1:
            km = float(np.sum(np.abs(d.v_bar)) * period / 1000)
            ax.set_title(f"{n[:10].replace('2011_', '')} / {n[-4:]}\n{role(cfg, n)}, {km:.1f} km", fontsize=fsz(6))
            ax.set_xticks([]); ax.set_yticks([])
        else:
            ax.set_title(n.replace("2011_", "").replace("_drive_", "\n"), fontsize=fsz(6))
            ax.tick_params(labelsize=fsz(5))
    save(fig, figd, "fig_ds_trajectories")

    # ---- 4. LiDAR vs IMU/GPS motion agreement (benign) ----
    allb = pd.concat(benign.values())
    vh, wh = allb.v_hat * ratio, allb.w_hat * ratio
    fig, ax = plt.subplots(2, 2, figsize=(7.2, 4.6)) if SCALE > 1 else plt.subplots(1, 4, figsize=(7.2, 2.0))
    ax = np.ravel(ax)
    ax[0].hexbin(allb.v_bar, vh, gridsize=50, bins="log", cmap="Greys", mincnt=1)
    lim = [min(allb.v_bar.min(), vh.min()), max(allb.v_bar.max(), vh.max())]
    ax[0].plot(lim, lim, "r--", lw=0.6); ax[0].set_xlabel("IMU/GPS speed [m/s]"); ax[0].set_ylabel("LiDAR speed [m/s]")
    ax[1].hist(vh - allb.v_bar, bins=80, color="0.4"); ax[1].set_xlabel("speed difference [m/s]"); ax[1].set_ylabel("frames")
    ax[2].hexbin(np.degrees(allb.w_bar), np.degrees(wh), gridsize=50, bins="log", cmap="Greys", mincnt=1)
    lw = np.degrees([allb.w_bar.min(), allb.w_bar.max()])
    ax[2].plot(lw, lw, "r--", lw=0.6); ax[2].set_xlabel("IMU yaw rate [deg/s]"); ax[2].set_ylabel("LiDAR yaw rate [deg/s]")
    ax[3].hist(np.degrees(wh - allb.w_bar), bins=80, color="0.4"); ax[3].set_xlabel("yaw-rate difference [deg/s]"); ax[3].set_ylabel("frames")
    if SCALE > 1:
        ax[3].set_xlim(-6, 6)
    r_v = np.corrcoef(allb.v_bar, vh)[0, 1]; r_w = np.corrcoef(allb.w_bar, wh)[0, 1]
    ax[0].set_title(f"corr {r_v:.3f}", fontsize=fsz(7)); ax[2].set_title(f"corr {r_w:.3f}", fontsize=fsz(7))
    save(fig, figd, "fig_ds_motion_agreement")

    # ---- 5. feature distributions: benign vs replay (uses the scored streams of step 2) ----
    sc_dir = os.path.join(cfg["results_dir"], "tables", "scores")
    thr_path = os.path.join(cfg["results_dir"], "tables", "thresholds.json")
    scored = {os.path.basename(f)[:-4]: pd.read_csv(f) for f in sorted(glob.glob(os.path.join(sc_dir, "*.csv")))}
    if scored:
        thr = float(np.median(pd.concat(scored.values()).groupby("fold")["thr::FreshScan"].first()))  # median over folds
        R = cfg["rq3"]["replay_age_s"]
        ben = pd.concat([v for k, v in scored.items() if k.endswith("__benign")])
        att = pd.concat([v[v.label == 1] for k, v in scored.items() if f"__replay_R{R}_" in k])
        feats = [("r_v", "speed residual"), ("r_w", "yaw-rate residual"),
                 ("one_minus_fit", "1 - ICP fitness"), ("rmse", "ICP inlier RMSE [m]"),
                 ("score::FreshScan", "score" if SCALE > 1 else "FreshScan score")]
        fig, ax = plt.subplots(1, len(feats), figsize=(7.2, 2.5 if SCALE > 1 else 1.9))
        for a, (c, lab) in zip(ax, feats):
            hi = np.nanpercentile(np.r_[ben[c], att[c]], 99.5)
            b = np.linspace(0, hi if hi > 0 else 1, 40)
            a.hist(np.clip(ben[c], 0, b[-1]), bins=b, density=True, histtype="stepfilled", color="0.75", label="benign")
            a.hist(np.clip(att[c], 0, b[-1]), bins=b, density=True, histtype="step", color="black", label=f"replay {R} s")
            a.set_yscale("log"); a.set_xlabel(lab, fontsize=fsz(7)); a.tick_params(labelsize=fsz(6))
            if c == "score::FreshScan":
                a.axvline(thr, color="tab:red", ls="--", lw=0.8)
        if SCALE > 1:
            h, l = ax[0].get_legend_handles_labels()
            fig.tight_layout(rect=(0, 0, 1, 0.84))
            fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
        else:
            ax[0].legend(fontsize=fsz(6))
        save(fig, figd, "fig_ds_features", layout=SCALE == 1)

        # ---- 6. replay example ----
        key = next((k for k in scored if f"__replay_R{R}_s0" in k), None)
        if key:
            d = scored[key]; t = d.slot.values * period
            fig, ax = plt.subplots(3, 1, figsize=(7.2, 5.6 if SCALE > 1 else 4.2), sharex=True)
            for a in ax:
                lab = d.label.values
                for s0, e0 in zip(np.where(np.diff(np.r_[0, lab]) == 1)[0], np.where(np.diff(np.r_[lab, 0]) == -1)[0]):
                    a.axvspan(t[s0], t[e0], color="0.88", lw=0)
            ax[0].plot(t, d.slot, color="0.6", ls="--", label="expected (fresh)")
            ax[0].plot(t, d.sigma, color="black", label="delivered frame")
            ax[0].set_ylabel("frame index"); ax[0].legend(loc="upper left")
            ax[1].plot(t, d.v_bar, color="black", label="IMU/GPS speed now")
            ax[1].plot(t, d.v_bar_src, color="tab:blue", lw=0.7, label="speed when delivered frame was captured")
            ax[1].set_ylabel("m/s"); ax[1].legend(loc="upper left")
            ax[2].plot(t, d["score::FreshScan"], color="black", label="FreshScan score")
            thr_d = float(d["thr::FreshScan"].iloc[0])
            ax[2].axhline(thr_d, color="tab:red", ls="--", lw=0.8, label="threshold")
            al = d.alarm.values == 1
            ax[2].plot(t[al], np.full(al.sum(), thr_d * 1.05), "|", color="tab:red", ms=6, label="alarm")
            sa = d.seam_alarm.values == 1
            ax[2].plot(t[sa], np.full(sa.sum(), thr_d * 2.0), "v", color="tab:blue", ms=4, label="seam alarm")
            ax[2].set_yscale("log"); ax[2].set_ylabel("score"); ax[2].set_xlabel("time [s]")
            if SCALE > 1:
                for a, nc in zip(ax, (2, 2, 4)):
                    a.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=nc, frameon=False, borderaxespad=0.1,
                             columnspacing=1.0, handlelength=1.6)
            else:
                ax[2].legend(loc="upper left")
                ax[0].set_title(key.replace("__", "  "), fontsize=fsz(8), loc="left")
            save(fig, figd, "fig_ds_replay_example")

    # ---- 7. bird's-eye view from saved samples ----
    samples = sorted(glob.glob(os.path.join(cfg["results_dir"], "dataset_samples", "*.npz")))
    if samples:
        z = np.load(samples[0]); name = os.path.basename(samples[0])[:-4]
        from scipy.spatial import cKDTree
        def overlap(a, b):
            d, _ = cKDTree(b).query(a, distance_upper_bound=fs_cfg["d_max"])
            return float(np.isfinite(d).mean())
        fs_cfg = cfg["freshscan"]
        print(f"  BEV {name} frame {int(z['index'])} at {float(z['speed']):.1f} m/s: points of the current scan with a "
              f"neighbor within {fs_cfg['d_max']:g} m: previous scan {100 * overlap(z['cur'], z['prev']):.1f}%, "
              f"scan from 10 s earlier {100 * overlap(z['cur'], z['old']):.1f}%")
        fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.9 if SCALE > 1 else 2.6))
        b = z["bev"]
        ax[0].scatter(b[:, 0], b[:, 1], s=0.05, c=b[:, 2], cmap="viridis", vmin=-2.5, vmax=2.0)
        ax[0].set_title("(a) one scan (color: height)" if SCALE > 1 else f"{name}\nframe {int(z['index'])}, {float(z['speed']):.1f} m/s", fontsize=fsz(7))
        ax[1].scatter(z["prev"][:, 0], z["prev"][:, 1], s=0.08, color="0.6", label="previous scan")
        ax[1].scatter(z["cur"][:, 0], z["cur"][:, 1], s=0.08, color="black", label="current scan")
        ax[1].set_title("(b) previous + current scan" if SCALE > 1 else "fresh: previous + current scan", fontsize=fsz(7))
        ax[2].scatter(z["old"][:, 0], z["old"][:, 1], s=0.08, color="tab:red", label="10 s old scan")
        ax[2].scatter(z["cur"][:, 0], z["cur"][:, 1], s=0.08, color="black", label="current scan")
        ax[2].set_title("(c) 10 s old + current scan" if SCALE > 1 else "replay seam: 10 s old + current scan", fontsize=fsz(7))
        for a in ax:
            a.set_aspect("equal"); a.set_xlim(-40, 40); a.set_ylim(-40, 40); a.tick_params(labelsize=fsz(6))
            if SCALE > 1:
                a.set_xlabel("x [m]")
        if SCALE > 1:
            ax[0].set_ylabel("y [m]")
        save(fig, figd, "fig_ds_bev")
    else:
        print("  (no dataset_samples/*.npz yet: the bird's-eye-view figure needs --save-samples during step 1)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/scripts/05_evaluate_checkpoint.py
"""Step 5 (any time later): evaluate from the saved checkpoint, without the raw KITTI scans.

Uses results/checkpoint/ (written by 02_evaluate.py) and the per-frame feature CSVs of step 1.
Lets you try other operating points without recomputing anything expensive:
    * other false-positive rates (the threshold is re-set on the stored calibration scores)
    * other alarm rules (m of the last n frames) and seam-alarm rates
and reports detection, delay and false alarms per method, replay age and test drive.

Outputs (results/tables/):
    checkpoint_eval.csv            one row per (method, target FPR, replay age)
    checkpoint_eval_per_drive.csv  FreshScan per test drive and replay age

Usage (from the repository root):
    python scripts/05_evaluate_checkpoint.py                         # FPR 1%, 0.5%, 0.1%
    python scripts/05_evaluate_checkpoint.py --fpr 0.001 --m 3 --n 5
    python scripts/05_evaluate_checkpoint.py --features other_results/features   # new data, same fit
"""

from __future__ import annotations

import argparse
import glob
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from common import load_config, out_dir  # noqa: E402
from freshscan.checkpoint import B1, load_checkpoint

METHODS = [B1, "B2: Seam only", "B3: Residuals only", "FreshScan"]


def events(alarm: np.ndarray) -> int:
    a = np.asarray(alarm, dtype=int)
    return int(((a == 1) & (np.r_[0, a[:-1]] == 0)).sum())


def window_delays(label: np.ndarray, alarm: np.ndarray) -> list:
    starts = np.where(np.diff(np.r_[0, label]) == 1)[0]
    ends = np.where(np.diff(np.r_[label, 0]) == -1)[0]
    out = []
    for s, e in zip(starts, ends):
        hit = np.where(alarm[s:e + 1])[0]
        out.append(int(hit[0]) if len(hit) else None)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--checkpoint", default=None, help="checkpoint folder (default: results/checkpoint)")
    ap.add_argument("--features", default=None, help="feature CSV folder (default: results/features)")
    ap.add_argument("--fpr", type=float, nargs="+", default=[0.01, 0.005, 0.001])
    ap.add_argument("--seam-fpr", type=float, default=None, help="seam-alarm rate (default: as calibrated)")
    ap.add_argument("--m", type=int, default=None)
    ap.add_argument("--n", type=int, default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    ck = load_checkpoint(args.checkpoint or os.path.join(cfg["results_dir"], "checkpoint"))
    fs = ck.meta["freshscan"]
    m_, n_ = args.m or fs["m"], args.n or fs["n"]
    feat_dir = args.features or os.path.join(cfg["results_dir"], "features")
    test_drives = ck.meta["test_drives"]
    raw = {}
    for f in sorted(glob.glob(os.path.join(feat_dir, "*.csv"))):
        drive, setting = os.path.basename(f)[:-4].split("__")
        if drive in test_drives:
            raw[(drive, setting)] = pd.read_csv(f)
    if not raw:
        raise FileNotFoundError(f"No test-drive feature CSVs in {feat_dir}")
    print(f"Checkpoint from {ck.meta['created']} | protocol {ck.meta['protocol']} | {len(raw)} streams | "
          f"alarm rule {m_} of {n_} + seam alarm")

    rows, per_drive = [], []
    for fpr in args.fpr:
        streams = {k: ck.fold_for(k[0]).score(df, fpr=fpr, m=m_, n=n_, seam_fpr=args.seam_fpr)
                   for k, df in raw.items()}
        benign = [k for k in streams if k[1] == "benign"]
        ages = sorted({int(k[1].split("_")[1][1:]) for k in streams if k[1].startswith("replay")})
        period = np.mean([ck.fold_for(k[0]).period for k in benign])
        minutes = sum(len(streams[k]) for k in benign) * period / 60.0
        for meth in METHODS:
            col, tcol = f"score::{meth}", f"thr::{meth}"
            ben = np.concatenate([streams[k][col].values for k in benign])
            ben_hit = np.concatenate([streams[k][col].values > streams[k][tcol].values for k in benign])
            acol = f"alarm::{meth}"
            fa = sum(events(streams[k][acol].values) for k in benign)
            for R in ages:
                keys = [k for k in streams if k[1].startswith(f"replay_R{R}_")]
                att = np.concatenate([streams[k].loc[streams[k].label == 1, col].values for k in keys])
                hits = np.concatenate([(streams[k][col] > streams[k][tcol]).values[streams[k].label.values == 1]
                                       for k in keys])
                delays = []
                for k in keys:
                    wd = window_delays(streams[k].label.values, streams[k][acol].values.astype(bool))
                    delays += wd
                    if meth == "FreshScan":
                        lab = streams[k].label.values == 1
                        per_drive.append(dict(target_fpr=fpr, drive=k[0], replay_age_s=R,
                                              tpr=float(np.mean(streams[k][col].values[lab] > streams[k][tcol].values[lab])),
                                              windows_detected_pct=100.0 * np.mean([d is not None for d in wd])))
                hit = [d for d in delays if d is not None]
                y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
                rows.append(dict(method=meth, target_fpr=fpr, replay_age_s=R,
                                 auroc=float(roc_auc_score(y, np.r_[ben, att])),
                                 benign_fpr_on_test=float(ben_hit.mean()), tpr=float(hits.mean()),
                                 windows_detected_pct=100.0 * len(hit) / max(len(delays), 1),
                                 median_delay_frames=float(np.median(hit)) if hit else float("nan"),
                                 p95_delay_frames=float(np.percentile(hit, 95)) if hit else float("nan"),
                                 false_alarm_events_per_min=fa / max(minutes, 1e-9)))
    tab = out_dir(cfg, "tables")
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(tab, "checkpoint_eval.csv"), index=False)
    pd.DataFrame(per_drive).groupby(["target_fpr", "drive", "replay_age_s"], as_index=False)[
        ["tpr", "windows_detected_pct"]].mean().to_csv(os.path.join(tab, "checkpoint_eval_per_drive.csv"), index=False)
    show = df[df.method == "FreshScan"][["target_fpr", "replay_age_s", "auroc", "tpr", "benign_fpr_on_test",
                                          "windows_detected_pct", "median_delay_frames", "p95_delay_frames",
                                          "false_alarm_events_per_min"]]
    print(show.round(3).to_string(index=False))
    print(f"Written to {tab}/checkpoint_eval.csv and checkpoint_eval_per_drive.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/configs/default.yaml
# FreshScan - experiment configuration
# All paths are relative to the repository root unless absolute.

kitti_root: data/kitti_raw          # where scripts/00_download_kitti.sh unpacks the drives
results_dir: results

# Drives (KITTI raw "synced + rectified"). Long drives are needed for replay ages up to 30 s.
# These long drives are a suggestion; any synced drive with >= 1000 frames works.
# Calibration drives are used ONLY to fit benign statistics and thresholds.
# Evaluation protocol: leave_one_drive_out (each test drive is calibrated on the benign
# streams of all other drives; no attack data) or fixed (calibrate once on calibration_drives).
protocol: leave_one_drive_out

calibration_drives:
  - 2011_09_30_drive_0020
  - 2011_09_30_drive_0033
  - 2011_10_03_drive_0042
test_drives:
  - 2011_09_30_drive_0018
  - 2011_09_30_drive_0027
  - 2011_09_30_drive_0034
  - 2011_10_03_drive_0027

# Replay attack settings
replay:
  ages_s: [5, 10, 30]       # replay age R in seconds
  win_min_s: 2.0
  win_max_s: 10.0
  gap_s: 10.0
  seeds: [0, 1, 2]

# FreshScan parameters. Tune on calibration drives only.
freshscan:
  period_s: 0.1             # LiDAR period Delta
  r_min: 3.0                # range crop [m]
  r_max: 60.0
  voxel: 0.5                # voxel size delta [m]
  d_max: 1.0                # ICP match distance [m]
  icp_iters: 50
  icp_backend: auto          # auto | cuda | open3d | numpy  (same result, different speed)
  fit_reset: 0.5            # restart ICP from identity if the previous fitness was below this
  v0: 3.0                   # speed offset of the speed residual [m/s]
  smooth_frames: 3          # motion checks use the mean of the last k slots (causal)
  v_min: 2.0                # motion checks abstain below this reference speed [m/s] (P3)
  lidar_period_s: auto      # real scan period; "auto" = median header step on benign calibration drives
  w0: 0.05                  # yaw-rate offset [rad/s]: yaw residual is relative, like the speed residual
  seam_alarm_fpr: 0.001     # one scan whose ICP fit is this unusual (0.1% of benign scans) raises an alarm at once
  percentile: 99.0          # benign percentile used to normalize each feature
  target_fpr: 0.01          # threshold eta is set for this benign false-positive rate
  m: 2                      # m-of-n alarm rule
  n: 3

# Bins for the RQ3 analysis
rq3:
  replay_age_s: 10
  dv_bins: [0.5, 2.0]       # |v(t) - v(t-R)| edges [m/s]
  speed_bins: [1.0, 10.0]   # current speed edges [m/s]

In [ ]:
#@title 5. Make the Colab config file
import yaml

def make_config(src_name, out_name, kitti_root):
    cfg = yaml.safe_load(open(f"{CODE}/configs/{src_name}"))
    cfg["kitti_root"], cfg["results_dir"] = kitti_root, RESULTS
    cfg["freshscan"]["icp_backend"] = ICP_BACKEND
    with open(f"{CODE}/configs/{out_name}", "w") as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    return f"configs/{out_name}"

CONFIG = make_config("default.yaml", "colab.yaml", KITTI)
CFG = yaml.safe_load(open(f"{CODE}/{CONFIG}"))
print("Calibration drives:", CFG["calibration_drives"])
print("Test drives:       ", CFG["test_drives"])

sys.path.insert(0, CODE)
from freshscan import registration as reg
print("ICP backend for this run:", reg.select_backend(ICP_BACKEND))

## 6. Step 1: compute features for every drive
For each KITTI drive, the notebook downloads the zip, unpacks only the LiDAR scans and IMU/GPS files, runs FreshScan (C1–C3) on the normal stream and on every replayed stream, and then deletes the scans. When a drive is finished, a `_done_<drive>.txt` file goes into the results folder so it is skipped next time.

In [ ]:
#@title 6. Step 1 (longest step)
feat_dir = f"{RESULTS}/features"
os.makedirs(feat_dir, exist_ok=True)

def kitti_url(name):
    # Same URL pattern as KITTI's official raw_data_downloader.sh
    return f"https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{name}/{name}_sync.zip"

def get_drive(name):
    base = f"{KITTI}/{name[:10]}/{name}_sync"
    if os.path.isdir(f"{base}/velodyne_points/data") and os.path.isdir(f"{base}/oxts/data"):
        print(f"[{name}] scans already on disk"); return base
    zp = f"{ZIPS}/{name}_sync.zip"
    manual = f"{OUT}/kitti_zips/{name}_sync.zip"     # optional: zips you put in Drive yourself
    if os.path.exists(manual):
        print(f"[{name}] using {manual}"); shutil.copy(manual, zp)
    else:
        try:
            sh(f"wget -c -nv -O {zp} {kitti_url(name)}", cwd="/content")
        except RuntimeError:
            raise RuntimeError(
                f"Download of {name} failed. Download '{name}_sync.zip' (synced+rectified data) by hand "
                f"from the KITTI raw data page, put it in {OUT}/kitti_zips/, and run this cell again.")
    # unpack only what FreshScan needs (no camera images)
    sh(f"unzip -q -o {zp} '*/velodyne_points/*' '*/oxts/*' -d {KITTI}", cwd="/content")
    os.remove(zp)
    return base

def free_gb(path="/content"):
    return shutil.disk_usage(path).free / 1e9

drives = CFG["calibration_drives"] + CFG["test_drives"]
for k, name in enumerate(drives, 1):
    done = f"{feat_dir}/_done_{name}.txt"
    if os.path.exists(done):
        print(f"({k}/{len(drives)}) [{name}] already finished, skipped"); continue
    t0 = time.time()
    print(f"\n({k}/{len(drives)}) [{name}]  free disk: {free_gb():.0f} GB")
    base = get_drive(name)
    sh(f"python scripts/01_compute_features.py --config {CONFIG} --drives {name}")
    # keep a few scans of this drive for the bird's-eye-view figure (before the scans are deleted)
    sh(f"python scripts/04_visualize_dataset.py --config {CONFIG} --save-samples {name}")
    with open(done, "w") as fh:
        fh.write(time.strftime("%Y-%m-%d %H:%M:%S"))
    if DELETE_SCANS_AFTER_EACH_DRIVE:
        shutil.rmtree(base, ignore_errors=True)
    print(f"[{name}] finished in {(time.time() - t0) / 60:.1f} min")
print("\nStep 1 done. Feature files:", len(glob.glob(f"{feat_dir}/*.csv")))

In [ ]:
#@title 7. Steps 2-4: evaluate, save the checkpoint, make the figures
sh(f"python scripts/02_evaluate.py --config {CONFIG}")        # also writes checkpoint/
sh(f"python scripts/03_make_figures.py --config {CONFIG}")
sh(f"python scripts/04_visualize_dataset.py --config {CONFIG}")

In [ ]:
#@title 8. Show the results
import pandas as pd
from IPython.display import display, Image, Markdown

tab_dir = f"{RESULTS}/tables"
for name, title in [("dataset_summary", "Dataset summary"),
                    ("rq1_detection", "RQ1: detection quality"),
                    ("rq2_delay", "RQ2: detection delay and false alarms"),
                    ("rq3_analysis", "RQ3: when it works or fails"),
                    ("rq4_runtime", "RQ4: run time")]:
    path = f"{tab_dir}/{name}.csv"
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(pd.read_csv(path).round(3))
display(Markdown("## Result figures (03_make_figures.py)"))
for name, title in [("fig_bev", "Bird's-eye view of a replay seam"),
                    ("fig_dataset", "The drives (speed, turning, driven paths)"),
                    ("fig_motion", "LiDAR (ICP) motion vs IMU/GPS motion on normal driving"),
                    ("fig_features", "Each check on normal vs replayed frames"),
                    ("fig_roc", "ROC curves (replay age 10 s)"),
                    ("fig_replay_drive", "One whole test drive under replay"),
                    ("fig_timeseries", "Zoom on one replay window")]:
    path = f"{RESULTS}/figures/{name}.png"
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(Image(path, width=760))
if os.path.exists(f"{tab_dir}/summary_stats.json"):
    display(Markdown("### Other numbers (correlations, errors, overlap)"))
    print(open(f"{tab_dir}/summary_stats.json").read())

display(Markdown("## Dataset plots (04_visualize_dataset.py)"))
if os.path.exists(f"{tab_dir}/dataset_drives.csv"):
    display(Markdown("### Drives used (one row per drive)"))
    display(pd.read_csv(f"{tab_dir}/dataset_drives.csv"))
for name, title in [("fig_ds_overview", "Speed (black) and yaw rate (blue) of every drive; dotted line = abstain speed"),
                    ("fig_ds_distributions", "Speed and yaw-rate distributions, calibration vs test drives"),
                    ("fig_ds_trajectories", "Driven paths (integrated from IMU/GPS), green dot = start"),
                    ("fig_ds_motion_agreement", "LiDAR (ICP) motion vs IMU/GPS motion on normal driving"),
                    ("fig_ds_features", "Each check on normal vs replayed scans (red line = alarm threshold)"),
                    ("fig_ds_replay_example", "One replay stream: what was delivered, the speeds, the score and the alarms"),
                    ("fig_ds_bev", "Bird's-eye view: a fresh scan pair vs a 10 s old scan next to the current one")]:
    path = f"{RESULTS}/figures/dataset/{name}.png"
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(Image(path, width=760))

In [ ]:
#@title 9. Download all results as one ZIP
archive = shutil.make_archive("/content/freshscan_results", "zip", RESULTS)
print("Saved", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
print("Results are also in", RESULTS)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## 10. Evaluate again later from the checkpoint
Can be used after any finished run, also in a new session: run cells 1-5, then this cell. It reads `checkpoint/` and `features/` from your Drive. Nothing is downloaded and ICP is not run again, so it takes seconds.

* **False-positive rates:** each one sets a new alarm threshold on the stored calibration scores. A lower rate means fewer false alarms but later or missed detections.
* **Alarm rule:** an alarm fires when at least `M` of the last `N` scans are above the threshold.
* **Outputs:** `tables/checkpoint_eval.csv` (per method, false-positive rate and replay age) and `tables/checkpoint_eval_per_drive.csv`.

To change other settings, such as the 3-scan averaging or the 2 m/s cutoff, edit `configs/default.yaml` in cell 4 and rerun cell 7. That re-calibrates from the stored features and writes a new checkpoint, still without ICP.

In [ ]:
#@title 10. Evaluation from the checkpoint  { display-mode: "form" }
FALSE_POSITIVE_RATES = "0.01 0.005 0.002 0.001"  #@param {type:"string"}
ALARM_M = 2  #@param {type:"integer"}
ALARM_N = 3  #@param {type:"integer"}
sh(f"python scripts/05_evaluate_checkpoint.py --config {CONFIG} --fpr {FALSE_POSITIVE_RATES} "
   f"--m {ALARM_M} --n {ALARM_N}")
import pandas as pd
from IPython.display import display
display(pd.read_csv(f"{RESULTS}/tables/checkpoint_eval.csv").round(3))

## Output files
| Result | File in the results folder |
|---|---|
| RQ1: AUROC and TPR at 1% false positives | `tables/rq1_detection.csv` |
| RQ2: detection delay, windows detected, false alarms per minute | `tables/rq2_delay.csv` |
| RQ3: detection by seam, speed difference and speed | `tables/rq3_analysis.csv` |
| RQ4: time per frame | `tables/rq4_runtime.csv` |
| Other thresholds (cell 10) | `tables/checkpoint_eval.csv` |
| Drives (frames, minutes, km, speeds) | `tables/dataset_drives.csv`, `tables/dataset_summary.csv` |
| Figures (vector PDF) | `figures/fig_*.pdf` |
| Calibration | `checkpoint/` |
| Hardware per drive | `run_info/*.json` |